# Digifly Phase 2 – Unified Simulation Runner (Full Phase-2-Master Controls)

This notebook is a **drop-in replacement** for `notebooks/run_simulation.ipynb`.

It supports all simulation modes:
- **single neuron**
- **custom network** (user-provided edges file)
- **hemilineage network** (e.g., 09A)

Only the configuration cells require editing before running top-to-bottom.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
import json

def _safe_cwd() -> Path:
    """
    Return a usable working directory even if the kernel's cwd was deleted/moved.
    """
    try:
        return Path(os.getcwd())
    except FileNotFoundError:
        home = os.environ.get("HOME")
        if home:
            return Path(home)
        return Path("/tmp")

def _find_repo_root() -> Path:
    """
    Find the Phase 2 repo root by locating a folder that contains 'digifly/'.
    Set DIGIFLY_PHASE2_ROOT when the notebook server starts outside the repository.
    """
    candidates: list[Path] = []

    env_root = os.environ.get("DIGIFLY_PHASE2_ROOT", "").strip()
    if env_root:
        candidates.append(Path(env_root))

    cwd = _safe_cwd()
    candidates.append(cwd)

    try:
        candidates.append(Path.cwd())
    except Exception:
        pass

    try:
        candidates.append(Path(__file__).resolve().parent)
    except NameError:
        pass

    seen = set()
    expanded: list[Path] = []
    for candidate in candidates:
        try:
            candidate = candidate.expanduser().resolve()
        except Exception:
            candidate = candidate.expanduser()
        if str(candidate) in seen:
            continue
        seen.add(str(candidate))
        expanded.append(candidate)

    for base in expanded:
        for p in [base] + list(base.parents):
            if (p / "digifly").exists():
                return p

    raise RuntimeError(
        "Could not locate the Phase 2 root folder containing /digifly.\n"
        f"Safe cwd was: {_safe_cwd()}\n"
        "Start Jupyter from the Phase 2 folder, or set DIGIFLY_PHASE2_ROOT."
    )

REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Force-refresh local digifly modules so notebook reruns pick up file edits.
import importlib
import digifly.phase2.config.loader as _cfg_loader
import digifly.phase2.neuron_build.timing as _timing_mod
import digifly.phase2.neuron_build.network as _network_mod
import digifly.phase2.neuron_build.gaps as _gaps_mod
import digifly.phase2.util.save as _save_util
import digifly.phase2.neuron_build.builders as _builders
import digifly.phase2.walking.runner as _walker_runner
import digifly.phase2.api as _phase2_api

# Reload dependency order: low-level modules first, then orchestrators.
for _m in (_timing_mod, _network_mod, _gaps_mod, _save_util, _cfg_loader, _builders, _walker_runner, _phase2_api):
    importlib.reload(_m)

from digifly.phase2.neuron_build.gaps import (
    gap_pair_ohmic,
    gap_pair_rectifying,
    gap_pair_directed,
)
from digifly.phase2.api import build_config, run_walking_simulation
print(f"[ok] REPO_ROOT = {REPO_ROOT} (digifly modules reloaded: timing/network/gaps/save/builders/runner/api)")


## 1) Required paths (Phase 2 Master compatible)

In [ ]:
# REQUIRED: points at the export_swc folder produced by Phase 1.
# Set DIGIFLY_SWC_DIR before launching Jupyter, or edit SWC_DIR here.
# Example:
#   SWC_DIR = r"/path/to/export_swc"
SWC_DIR = os.environ.get("DIGIFLY_SWC_DIR", "").strip()
if not SWC_DIR:
    SWC_DIR = str((REPO_ROOT / "data" / "export_swc").resolve())

# Optional alternate SWC root for reduced morphologies (leave None to use SWC_DIR)
MORPH_SWC_DIR = os.environ.get("DIGIFLY_MORPH_SWC_DIR", "").strip() or None

# Optional override; if None, loader will use: SWC_DIR/../all_neurons_neuroncriteria_template.csv
MASTER_CSV = os.environ.get("DIGIFLY_MASTER_CSV", "").strip() or None

# Optional overrides; if None, loader will use:
#   EDGES_ROOT = SWC_DIR/edges
#   RUNS_ROOT  = SWC_DIR/hemi_runs
EDGES_ROOT = os.environ.get("DIGIFLY_EDGES_ROOT", "").strip() or None
RUNS_ROOT = os.environ.get("DIGIFLY_RUNS_ROOT", "").strip() or None

swc_path = Path(SWC_DIR).expanduser().resolve()
if not swc_path.exists():
    raise FileNotFoundError(
        "SWC_DIR does not exist. Generate/export SWCs in Phase 1, set DIGIFLY_SWC_DIR, "
        f"or edit SWC_DIR in this cell. Tried: {swc_path}"
    )

# Allow defaults.py to pick this up as a default (user overrides still win)
os.environ["DIGIFLY_SWC_DIR"] = str(swc_path)
print("SWC_DIR =", swc_path)
if MORPH_SWC_DIR is not None:
    print("MORPH_SWC_DIR =", Path(MORPH_SWC_DIR).expanduser().resolve())


## 1B) Optional: Build Reduced SWC Dataset (Jupyter, non-destructive)

In [ ]:
# Build reduced SWCs from SWC_DIR into a separate tree (never edits originals).
# Keep this OFF unless a reduction job should run from the notebook.
BUILD_REDUCED_SWC = False

# Output root for reduced SWCs
REDUCE_OUTPUT_ROOT = str(Path(SWC_DIR).expanduser().resolve().parent / "export_swc_reduced" / "v1")

# Parallelism + targeting
REDUCE_WORKERS = 32
REDUCE_LIMIT = 0          # 0 => all SWCs
REDUCE_MATCH = None       # optional substring filter on SWC path
REDUCE_OVERWRITE = False
REDUCE_WRITE_MAP = False  # write per-neuron old->new node map CSV

# Reduction criteria
REDUCE_MAX_PATH_UM = 8.0
REDUCE_MAX_TURN_DEG = 35.0
REDUCE_MAX_DIAM_REL = 0.35
REDUCE_PROTECT_SYNAPSES = True
REDUCE_MAX_SYN_POINTS = 2000

if BUILD_REDUCED_SWC:
    from digifly.tools.reduce_swc_dataset import main as reduce_swc_main

    reduce_args = [
        "--input-root", str(Path(SWC_DIR).expanduser().resolve()),
        "--output-root", str(Path(REDUCE_OUTPUT_ROOT).expanduser().resolve()),
        "--workers", str(int(REDUCE_WORKERS)),
        "--max-path-um", str(float(REDUCE_MAX_PATH_UM)),
        "--max-turn-deg", str(float(REDUCE_MAX_TURN_DEG)),
        "--max-diam-rel", str(float(REDUCE_MAX_DIAM_REL)),
        "--max-syn-points", str(int(REDUCE_MAX_SYN_POINTS)),
    ]
    if int(REDUCE_LIMIT) > 0:
        reduce_args += ["--limit", str(int(REDUCE_LIMIT))]
    if REDUCE_MATCH:
        reduce_args += ["--match", str(REDUCE_MATCH)]
    if REDUCE_OVERWRITE:
        reduce_args += ["--overwrite"]
    if REDUCE_WRITE_MAP:
        reduce_args += ["--write-map"]
    if not bool(REDUCE_PROTECT_SYNAPSES):
        reduce_args += ["--no-protect-synapses"]

    rc = reduce_swc_main(reduce_args)
    if int(rc) != 0:
        raise RuntimeError(f"SWC reduction failed with exit code {rc}")

    MORPH_SWC_DIR = str(Path(REDUCE_OUTPUT_ROOT).expanduser().resolve())
    print("[reduce] done. MORPH_SWC_DIR set to:", MORPH_SWC_DIR)
else:
    print("[reduce] skipped (BUILD_REDUCED_SWC=False)")


## 2) Choose simulation mode + selection

In [ ]:
# REQUIRED (no implicit default): 'single' | 'custom' | 'hemilineage'
MODE = "custom"


### 2A) Hemilineage selection

In [ ]:
# Used only if MODE == 'hemilineage'
HEMI_LABEL = "09A"

# Seeds behavior:
#   - None => simulate ALL seeds (default behavior)
#   - list[int] => simulate only those seeds
SEEDS = None


### 2B) Single neuron selection

In [ ]:
# Used only if MODE == 'single'
NEURON_ID = 10000


### 2C) Custom network selection

In [ ]:
# Used only if MODE == 'custom'
# Neurons to load (cells created from SWC).
NEURON_IDS = [10000, 10002, 10068, 11446, 11654, 10110]
STIM_NEURON_IDS = [10000, 10002]

# Optional chemical-only downstream (postsynaptic) additions into the current custom circuit.
# Keys are existing loaded source neurons; values are added postsynaptic target neurons.
# Leave empty for the original run_simulation behavior.
CHEM_ONLY_OUTPUTS_BY_SOURCE = {}
CHEM_ONLY_CONNECTION_DIRECTION = "source_to_postsyn"
# Legacy alias retained for compatibility with older helper code.
CHEM_ONLY_INPUTS_BY_TARGET = CHEM_ONLY_OUTPUTS_BY_SOURCE

# Flatten the added chemical-only neurons and append them to NEURON_IDS (deduped, order kept).
if MODE == "custom" and isinstance(CHEM_ONLY_INPUTS_BY_TARGET, dict) and CHEM_ONLY_INPUTS_BY_TARGET:
    _base_ids = [int(x) for x in NEURON_IDS]
    _extra_added = []
    _anchor_source_ids = []
    for _src, _post_list in CHEM_ONLY_INPUTS_BY_TARGET.items():
        _anchor_source_ids.append(int(_src))
        for _nid in (_post_list or []):
            _extra_added.append(int(_nid))

    _missing_sources = sorted(set(_anchor_source_ids) - set(_base_ids))
    if _missing_sources:
        print(
            "[chem-only] warning: source ids not currently in NEURON_IDS: "
            f"{_missing_sources}. If intentional, add them to NEURON_IDS too."
        )

    _seen = set()
    _merged = []
    for _nid in _base_ids + _extra_added:
        if _nid in _seen:
            continue
        _seen.add(_nid)
        _merged.append(_nid)
    NEURON_IDS = _merged

    print(
        f"[chem-only] added {len(set(_extra_added))} downstream/postsynaptic neurons "
        f"for sources {sorted(set(_anchor_source_ids))}; total loaded NEURON_IDS={len(NEURON_IDS)}"
    )

# Choose custom-edge source strategy:
#   - "path"  => use EDGES_PATH directly (legacy behavior)
#   - "cache" => use local SQLite edge cache + fast subset query (recommended)
CUSTOM_EDGE_SOURCE =  "path" #"cache"

# Used only when CUSTOM_EDGE_SOURCE == "path".
# Leave blank to reuse/build a named edge set for the current NEURON_IDS.
EDGES_PATH = ""
EDGE_SET_NAME = ""
EDGE_NEUPRINT_DATASET_FAMILY = ""
EDGE_NEUPRINT_DATASET_VERSION = ""
EDGE_NEUPRINT_DATASET = ""
EDGES_REGISTRY_ROOT = None
AUTO_BUILD_EDGE_SET_IF_MISSING = True
BUILD_EDGES_WORKERS = 16
CUSTOM_EDGE_EXPANSION_MODE = "none"  # 'none' | 'immediate_motor_postsynaptic'
PHASE1_FALLBACK_ENABLED = True
PHASE1_UPSAMPLE_NM = 2000.0
PHASE1_MIN_CONF = 0.4
PHASE1_BATCH_SIZE = 10000

# Cache controls (used when CUSTOM_EDGE_SOURCE == "cache")
# query mode:
#   - loaded_subgraph: edges among NEURON_IDS
#   - seed_io_1hop   : seeds + all direct inputs/outputs + edges among them
EDGE_CACHE_QUERY_MODE = "loaded_subgraph"
EDGE_CACHE_INCLUDE_LOADED = True
EDGE_CACHE_MAX_NODES = 5000
EDGE_CACHE_MAX_ROWS = None

# Large-run perf preset (applied later in the build/run cell only for
# custom + cache + seed_io_1hop runs).
FAST_1HOP_MODE = False
FAST_1HOP_COALESCE_MODE = "pair"   # "pair" fastest; "site" is more spatially faithful
FAST_1HOP_DISABLE_GEOM_DELAY = True
FAST_1HOP_DISABLE_CVODE = True
FAST_1HOP_FORCE_CORENEURON = True
FAST_1HOP_FORCE_GPU = True
FAST_1HOP_PROGRESS_CHUNK_MS = 0.5

EDGE_CACHE_BUILD_IF_MISSING = True
EDGE_CACHE_FORCE_REBUILD = False
# build_mode:
#   - from_edges_files: build from existing *_from_synapses.* files in <SWC_DIR>/edges (fast)
#   - from_synapses_csv: build directly from all *_synapses_new.csv under SWC_DIR (slow, most complete)
EDGE_CACHE_BUILD_MODE = "from_synapses_csv"
EDGE_CACHE_DB_PATH = None
# Optional explicit source files list. None => auto-discover *_from_synapses* under <SWC_DIR>/edges.
EDGE_CACHE_SOURCE_PATHS = None

# Seeds behavior:
#   - None => stimulate ALL loaded neurons (default)
#   - list[int] => stimulate only those neurons (subset is allowed)
if MODE == "custom":
    SEEDS = [int(x) for x in STIM_NEURON_IDS]
    _loaded = set(int(x) for x in NEURON_IDS)
    _stim   = set(SEEDS)
    missing = sorted(_stim - _loaded)
    if missing:
        raise ValueError(f"STIM_NEURON_IDS must be a subset of NEURON_IDS. Missing from NEURON_IDS: {missing}")
    if CUSTOM_EDGE_SOURCE not in {"path", "cache"}:
        raise ValueError(f"CUSTOM_EDGE_SOURCE must be 'path' or 'cache', got: {CUSTOM_EDGE_SOURCE}")
    EDGE_CACHE_ENABLED = bool(CUSTOM_EDGE_SOURCE == "cache")
    if CUSTOM_EDGE_SOURCE == "path" and (not EDGES_PATH) and (not AUTO_BUILD_EDGE_SET_IF_MISSING):
        raise ValueError("CUSTOM_EDGE_SOURCE='path' requires EDGES_PATH unless AUTO_BUILD_EDGE_SET_IF_MISSING=True")
else:
    SEEDS = None
    EDGE_CACHE_ENABLED = False



## 3) Run identity + simulation clock

In [ ]:
RUN_ID = "run_6n_baseline_nocoalesce"
RUN_NOTES = "2-seed seed_io_1hop neighborhood run (auto-drop missing SWC IDs)"

# Simulation time controls
TSTOP_MS = 10.0
DT_MS = 0.025
CELSIUS_C = 22.0


## 4) NEURON / CoreNEURON runtime knobs (Phase 2 Master category 1)

In [ ]:
# Progress / UI
PROGRESS = True
USE_TQDM = True
PROGRESS_CHUNK_MS = 0.5

# Threading (general + CoreNEURON)
THREADS = 16                   # ParallelContext thread target for threaded simulation setup
# Preferred fast path: CoreNEURON + GPU (disable CVODE for this).
ENABLE_CORENEURON = True
CORENEURON_GPU = True
CORENEURON_NTHREAD = 8         # CoreNEURON thread target when supported

# CVODE variable-step controls (mutually exclusive with CoreNEURON)
CVODE_ENABLED = False
CVODE_ATOL = 1e-3
CVODE_RTOL = None              # optional; set float to enable relative tolerance
CVODE_MAXSTEP_MS = None        # optional cap for variable-step max dt (ms)
if ENABLE_CORENEURON and CVODE_ENABLED:
    print("[cfg] Both CoreNEURON and CVODE were enabled; disabling CVODE (CoreNEURON preferred).")
    CVODE_ENABLED = False
CVODE = {
    "enabled": bool(CVODE_ENABLED),
    "atol": float(CVODE_ATOL),
    "rtol": (float(CVODE_RTOL) if CVODE_RTOL is not None else None),
    "maxstep_ms": (float(CVODE_MAXSTEP_MS) if CVODE_MAXSTEP_MS is not None else None),
}

# NOTE: Some of these are only used when the runner/NEURON integration supports them.



## 5) IO knobs (Phase 2 Master category 6)

In [ ]:
# Output writing parallelism
IO_WORKERS = 8                 # output-writing worker target
IO_PARALLEL = None             # None | 'thread' | 'process' | 'auto'


## 6) Graph / wiring robustness knobs (Phase 2 Master category 2)

In [ ]:
# Wiring controls that matter for large graphs
# Set True only for debugging soma-clamp wiring; False preserves spatial synapse placement
WIRE_FORCE_SOMA = False

COALESCE_SYNS = False
COALESCE_MODE = "none"
MIN_WEIGHT_uS = 0.0

# Coalescing guardrails (Master keys)
COALESCE_GUARD_W_MED_uS = None
COALESCE_GUARD_DROP = None



## 7) Edges timing defaults + spatial epsilon (Phase 2 Master category 3)

In [ ]:
# These are used if an edges file is missing timing columns or has NaNs.
# Keep these at Copy2-like values to avoid accidental conductance blowups.
DEFAULT_WEIGHT_uS = 6e-6
DEFAULT_DELAY_MS  = 1.0
GLOBAL_WEIGHT_SCALE = 1.0
# Geometric-delay controls (used when USE_GEOM_DELAY=True)
GLOBAL_BASE_RELEASE_DELAY_MS = 0.40
GLOBAL_VEL_UM_PER_MS = 1500.0
USE_GEOM_DELAY = True
SYN_TAU1_MS        = 0.5
SYN_TAU2_MS        = 3.0
SYN_E_REV_MV       = 0.0

# Spatial epsilon used for site mapping / co-location tolerances when supported by the pipeline.
EPSILON_UM = None


## 8) AIS mapping knobs (Phase 2 Master category 4)

In [ ]:
# Keep parity with Phase 2 Master AIS behavior; allow future dataset changes.
AIS_MIN_DIST_UM = None
AIS_MIN_XLOC = None


## 9) Biophysics (global + Master-compatible dicts) (Phase 2 Master category 5)

In [ ]:
# ============================================================
# Biophysics defaults tuned to match stable Copy2-like AP shapes
# ============================================================

# Global equilibrium potentials / rest / v-init
V_REST_MV = -65.0
V_INIT_MV = -65.0

# Daniel/Copy2-compatible ion reversal defaults
E_NA_MV = 65.0
E_K_MV  = -74.0
E_L_MV  = V_REST_MV

# Passive membrane defaults
PASSIVE_GLOBAL = {
    "Ra": 35.4,
    "cm": 1.0,
    "g_pas": 1e-4,
}

# IMPORTANT: keep HH_GLOBAL off so branch/soma dictionaries stay distinct.
# Setting HH_GLOBAL to a single strong profile can create depolarized plateaus.
HH_GLOBAL = None

# Master-style compartment-specific HH profiles
# At celsius_C=22.0, postsynaptic cells need a stronger active profile to fire
# from the reconstructed synaptic drive without extreme global weight scaling.
PRE_SOMA_HH = {"gnabar": 0.12, "gkbar": 0.036, "gl": 3e-4, "el": E_L_MV}
PRE_BRANCH_HH = {"gnabar": 0.02, "gkbar": 0.01, "gl": 1e-4, "el": E_L_MV}
POST_SOMA_HH = {"gnabar": 0.20, "gkbar": 0.03, "gl": 3e-4, "el": E_L_MV}
POST_BRANCH_HH = {"gnabar": 0.05, "gkbar": 0.008, "gl": 1e-4, "el": E_L_MV}

# Postsynaptic activation policy (Master key)
POST_ACTIVE = True


## 10) Stimulation (IClamp + optional negative pulse)

In [ ]:
# IClamp as the primary stimulation method
# For celsius_C=22.0, use a softer/longer AIS pulse to avoid a hard onset jump
# while still reliably initiating seed spikes.
ICLAMP = {
    "amp_nA": 0.6,
    "delay_ms": 2.0,
    "dur_ms": 0.3,
    "location": "ais",   # default 'ais'; other options may be supported by the pipeline
}

# Optional negative pulse (present but disabled by default)
NEG_PULSE = {
    "enabled": False,
    "amp_nA": -1.0,
    "delay_ms": 0.0,
    "dur_ms": 50.0,
}

# Optional frequency-based pulse train (disabled by default).
# Pulses repeat at freq_hz and are clipped to TSTOP_MS (or stop_ms if set).
PULSE_TRAIN = {
    "enabled": False,
    "freq_hz": 100.0,
    "amp_nA": ICLAMP["amp_nA"],
    "delay_ms": ICLAMP["delay_ms"],
    "dur_ms": ICLAMP["dur_ms"],
    "location": ICLAMP.get("location", "ais"),
    "stop_ms": None,
    "max_pulses": None,
    "include_base_iclamp": False,
}


## 10B) Gap Junction Configuration (ohmic + rectifying)

In [ ]:
# Gap-junction configuration for GF pathway testing.
#
# Recommended start for latency tuning:
#   GAP_G_US = 0.001 (1 nS), then sweep [0.001, 0.002, 0.005].
#
# Modes:
#   GAP_MODE = 'ohmic'      -> symmetric electrical coupling (A <-> B)
#   GAP_MODE = 'rectifying' -> directional coupling, controlled by GAP_RECTIFY_DIRECTION
#
# mechanisms_dir should point to a folder NEURON can load that includes Gap.mod
# (either the mechanism build dir like .../arm64, or its parent if load_mechanisms can resolve it).
GAP_ENABLED = True
GAP_MODE = "rectifying"                 # 'ohmic' | 'rectifying'
GAP_RECTIFY_DIRECTION = "a_to_b"   # used only when GAP_MODE='rectifying': 'a_to_b' | 'b_to_a'
GAP_DEFAULT_SITE = "ais"           # fallback only if synapse coords are unavailable
GAP_G_US = 0.001                    # 0.001 uS = 1 nS
GAP_ALL_SYNAPSES = True             # True => place one gap at every matching synapse coordinate
GAP_MAX_SYNAPSES = 1                # used only when GAP_ALL_SYNAPSES=False
GAP_MECHANISMS_DIR = str(Path(SWC_DIR).expanduser().resolve().parent)

# Core GF pairs
GF_GAP_PAIRS = [
    (10000, 10110),
    (10002, 10068),
]

# Optional extra pairs from prior Master copy behavior (enable if needed)
GF_GAP_PAIRS += [
    (10000, 11446),
    (10000, 11654),
    (10002, 11446),
    (10002, 11654),
]

if str(GAP_MODE).strip().lower() == "ohmic":
    _gap_pairs_cfg = [
        {
            **gap_pair_ohmic(a, b, g_uS=GAP_G_US, site_a=GAP_DEFAULT_SITE, site_b=GAP_DEFAULT_SITE),
            "placement": "synapse",
            "all_synapses": bool(GAP_ALL_SYNAPSES),
            "max_synapses": int(GAP_MAX_SYNAPSES),
        }
        for (a, b) in GF_GAP_PAIRS
    ]
elif str(GAP_MODE).strip().lower() == "rectifying":
    _gap_pairs_cfg = [
        {
            **gap_pair_rectifying(
                a,
                b,
                direction=GAP_RECTIFY_DIRECTION,
                g_uS=GAP_G_US,
                site_a=GAP_DEFAULT_SITE,
                site_b=GAP_DEFAULT_SITE,
            ),
            "placement": "synapse",
            "all_synapses": bool(GAP_ALL_SYNAPSES),
            "max_synapses": int(GAP_MAX_SYNAPSES),
        }
        for (a, b) in GF_GAP_PAIRS
    ]
else:
    raise ValueError(f"Unsupported GAP_MODE={GAP_MODE!r}. Use 'ohmic' or 'rectifying'.")

GAP = {
    "enabled": bool(GAP_ENABLED),
    "mechanisms_dir": GAP_MECHANISMS_DIR,
    "default_site": GAP_DEFAULT_SITE,
    "default_g_uS": float(GAP_G_US),
    "pairs": _gap_pairs_cfg,
}

print(f"[gap cfg] enabled={GAP['enabled']} mode={GAP_MODE} direction={GAP_RECTIFY_DIRECTION} g_uS={GAP_G_US}")
print(f"[gap cfg] placement=synapse all_synapses={GAP_ALL_SYNAPSES} max_synapses={GAP_MAX_SYNAPSES}")
print(f"[gap cfg] pairs={GF_GAP_PAIRS}")


## 11) Recording / outputs

In [ ]:
RECORD = {
    # 'none' | 'seeds' | 'all' | list[int]
    "soma_v": "seeds",
    "spikes": "seeds",
    "spike_thresh_mV": 20.0,  # use 20 mV to count true APs (not EPSP 0 mV crossings)
}


## 12) Silence controls (optional, off by default)

In [ ]:
SILENCE = {
    "enable": False,
    # 'set_gnabar' (set Na conductance to 0) OR 'clamp_rest' (hold at rest)
    "mode": "set_gnabar",
    "targets": [],           # list[int] bodyIds; (future: groups/classes)
    "gnabar": 0.0,
    "rest_mV": V_REST_MV,
}


## 13) Build CONFIG overrides and run

In [ ]:
# =========================
# Cell 13: Build CONFIG + Run
# =========================

# Shared Phase 2 notebook custom-circuit workflow (chem-only mapping + forced overlay fallback).
import importlib

try:
    import digifly.phase2.graph.custom_circuit_workflow as _phase2_custom_workflow_mod
    _phase2_custom_workflow_mod = importlib.reload(_phase2_custom_workflow_mod)
    from digifly.phase2.graph.custom_circuit_workflow import (
        apply_chem_only_workflow_to_cfg as _phase2_apply_chem_only_workflow_to_cfg,
        build_chem_only_pairs as _phase2_build_chem_only_pairs,
    )
except Exception as _phase2_custom_workflow_import_err:
    _phase2_apply_chem_only_workflow_to_cfg = None
    _phase2_build_chem_only_pairs = None
    print(f"[chem-only][warn] Phase 2 shared custom workflow import failed: {_phase2_custom_workflow_import_err}")

try:
    import digifly.phase2.graph.requested_edge_sets as _phase2_requested_edges_mod
    _phase2_requested_edges_mod = importlib.reload(_phase2_requested_edges_mod)
    from digifly.phase2.graph.requested_edge_sets import (
        DEFAULT_NEUPRINT_DATASET as _phase2_default_neuprint_dataset,
        default_neuprint_dataset_version as _phase2_default_neuprint_dataset_version,
        default_edges_registry_root as _phase2_default_edges_registry_root,
        ensure_named_edge_set as _phase2_ensure_named_edge_set,
        known_neuprint_dataset_versions as _phase2_known_neuprint_dataset_versions,
        normalize_neuprint_dataset_family as _phase2_normalize_neuprint_dataset_family,
        normalize_neuprint_dataset as _phase2_normalize_neuprint_dataset,
        resolve_neuprint_dataset_choice as _phase2_resolve_neuprint_dataset_choice,
    )
except Exception as _phase2_requested_edges_import_err:
    _phase2_default_neuprint_dataset = "manc:v1.2.1"
    _phase2_default_neuprint_dataset_version = lambda family=None: "v1.2.1"
    _phase2_default_edges_registry_root = None
    _phase2_ensure_named_edge_set = None
    _phase2_known_neuprint_dataset_versions = lambda family=None: ["v1.2.1"]
    _phase2_normalize_neuprint_dataset_family = lambda x: (str(x or "manc").strip().lower() or "manc")
    _phase2_normalize_neuprint_dataset = lambda x: x or _phase2_default_neuprint_dataset
    _phase2_resolve_neuprint_dataset_choice = lambda dataset=None, version=None: (
        f"{_phase2_normalize_neuprint_dataset_family(dataset)}:{str(version or _phase2_default_neuprint_dataset_version(dataset)).strip()}"
    )
    print(f"[edges][warn] requested-edge helper import failed: {_phase2_requested_edges_import_err}")

# ---- Helper: strip None so defaults apply cleanly ----
def _strip_none(d):
    if isinstance(d, dict):
        out = {}
        for k, v in d.items():
            vv = _strip_none(v)
            if vv is None:
                continue
            out[k] = vv
        return out
    return d

# ---- Helper: de-duplicate while preserving order ----
def _dedupe_preserve_order(seq):
    seen = set()
    out = []
    for x in seq:
        xi = int(x)
        if xi in seen:
            continue
        seen.add(xi)
        out.append(xi)
    return out

# ---- Helper: robustly update the record dict to record all spikes ----
def _apply_recording_policy(record_dict, loaded_ids, stim_ids, *, record_soma_for_all=False):
    """
    Ensure the recorder tracks spikes for ALL loaded neurons.
    Optionally record soma_v for all loaded neurons; default records soma_v only for stim neurons.
    This function tries to be schema-robust across slightly different key conventions.
    """
    rd = dict(record_dict) if isinstance(record_dict, dict) else {}

    loaded_ids = [int(x) for x in loaded_ids]
    stim_ids   = [int(x) for x in stim_ids]

    # ---- Spikes recording ----
    # Common record-key patterns:
    #   rd["spikes"] = "seeds"/"all"/[ids]
    #   rd["spike_ids"] = [...]
    #   rd["spike"] = {...}
    #   rd["spike"]["ids"] = [...]
    #
    # Force spike recording for all loaded neuron IDs.

    if "spikes" in rd:
        rd["spikes"] = list(loaded_ids)

    if "spike_ids" in rd:
        rd["spike_ids"] = list(loaded_ids)

    if "spike" in rd and isinstance(rd["spike"], dict):
        spike_cfg = dict(rd["spike"])
        if "ids" in spike_cfg:
            spike_cfg["ids"] = list(loaded_ids)
        if "neuron_ids" in spike_cfg:
            spike_cfg["neuron_ids"] = list(loaded_ids)
        if "cells" in spike_cfg:
            spike_cfg["cells"] = list(loaded_ids)
        rd["spike"] = spike_cfg

    # If none of the above keys exist, add the canonical "spikes" key.
    # build_config typically ignores unknown keys; the run path uses "record".
    if ("spikes" not in rd) and ("spike_ids" not in rd) and ("spike" not in rd):
        rd["spikes"] = list(loaded_ids)

    # ---- Soma voltage recording ----
    # Common patterns:
    #   rd["soma_v"] = "seeds"/"all"/[ids]
    #   rd["voltages"] = ...
    #   rd["v"] = ...
    #
    # Set soma_v when present; otherwise leave the record policy unchanged.

    target_v_ids = list(loaded_ids) if record_soma_for_all else list(stim_ids)

    if "soma_v" in rd:
        rd["soma_v"] = target_v_ids

    if "voltages" in rd and isinstance(rd["voltages"], dict):
        vcfg = dict(rd["voltages"])
        if "soma" in vcfg:
            vcfg["soma"] = target_v_ids
        if "soma_ids" in vcfg:
            vcfg["soma_ids"] = target_v_ids
        rd["voltages"] = vcfg

    if "v" in rd and isinstance(rd["v"], dict):
        vcfg = dict(rd["v"])
        if "soma" in vcfg:
            vcfg["soma"] = target_v_ids
        if "soma_ids" in vcfg:
            vcfg["soma_ids"] = target_v_ids
        rd["v"] = vcfg

    return rd

# -------------------------------------------------------------------
# Build selection dict (MODE must already be defined in earlier cells)
# -------------------------------------------------------------------
selection = {"mode": MODE, "label": None, "neuron_id": None, "neuron_ids": None}

if MODE == "hemilineage":
    selection["label"] = HEMI_LABEL

elif MODE == "single":
    selection["neuron_id"] = int(NEURON_ID)

elif MODE == "custom":
    # NEURON_IDS can contain duplicates; dedupe while preserving order
    selection["neuron_ids"] = _dedupe_preserve_order(NEURON_IDS)

else:
    raise ValueError("MODE must be: 'single' | 'custom' | 'hemilineage'")

# -------------------------------------------------------------------
# Stim targets: STIM_NEURON_IDS applies ONLY in MODE == "custom"
# -------------------------------------------------------------------
if MODE == "custom":
    # SEEDS = the neurons that get IClamp / stim drivers
    SEEDS = _dedupe_preserve_order(STIM_NEURON_IDS)

    loaded_ids = list(selection["neuron_ids"])
    missing = sorted(set(SEEDS) - set(loaded_ids))
    if missing:
        raise ValueError(
            "STIM_NEURON_IDS must be a subset of NEURON_IDS (loaded circuit).\n"
            f"Loaded neuron_ids: {loaded_ids}\n"
            f"Stim seeds:        {SEEDS}\n"
            f"Missing from load: {missing}"
        )

    _edges_path_candidate = Path(EDGES_PATH).expanduser().resolve() if EDGES_PATH else None
    _needs_named_edge_build = bool(
        CUSTOM_EDGE_SOURCE == "path"
        and AUTO_BUILD_EDGE_SET_IF_MISSING
        and (not EDGES_PATH or _edges_path_candidate is None or not _edges_path_candidate.exists())
    )
    if _needs_named_edge_build:
        if _phase2_ensure_named_edge_set is None:
            raise RuntimeError(
                "Custom edge auto-build was requested, but the requested-edge helper could not be imported."
            )
        if not str(EDGE_SET_NAME).strip():
            EDGE_SET_NAME = input(f"Edge-set name to save/reuse [{RUN_ID}_custom_edges]: ").strip()
        EDGE_SET_NAME = EDGE_SET_NAME or f"{RUN_ID}_custom_edges"
        _dataset_text = str(EDGE_NEUPRINT_DATASET).strip()
        if _dataset_text and (":" in _dataset_text):
            EDGE_NEUPRINT_DATASET = _phase2_resolve_neuprint_dataset_choice(_dataset_text)
            EDGE_NEUPRINT_DATASET_FAMILY = _phase2_normalize_neuprint_dataset_family(EDGE_NEUPRINT_DATASET)
            EDGE_NEUPRINT_DATASET_VERSION = str(EDGE_NEUPRINT_DATASET).split(":", 1)[1]
        else:
            _default_family = _phase2_normalize_neuprint_dataset_family(_phase2_default_neuprint_dataset)
            if _dataset_text and not str(EDGE_NEUPRINT_DATASET_FAMILY).strip():
                EDGE_NEUPRINT_DATASET_FAMILY = _dataset_text
            if not str(EDGE_NEUPRINT_DATASET_FAMILY).strip():
                EDGE_NEUPRINT_DATASET_FAMILY = input(
                    f"neuPrint dataset family for missing edges/SWCs [{_default_family}] (e.g. manc or male-cns): "
                ).strip()
            EDGE_NEUPRINT_DATASET_FAMILY = _phase2_normalize_neuprint_dataset_family(
                EDGE_NEUPRINT_DATASET_FAMILY or _default_family
            )
            _default_version = _phase2_default_neuprint_dataset_version(EDGE_NEUPRINT_DATASET_FAMILY)
            _known_versions = _phase2_known_neuprint_dataset_versions(EDGE_NEUPRINT_DATASET_FAMILY)
            _version_help = ", ".join(_known_versions) if _known_versions else "enter version manually"
            if not str(EDGE_NEUPRINT_DATASET_VERSION).strip():
                EDGE_NEUPRINT_DATASET_VERSION = input(
                    f"{EDGE_NEUPRINT_DATASET_FAMILY} version [{_default_version}] (known: {_version_help}): "
                ).strip()
            EDGE_NEUPRINT_DATASET = _phase2_resolve_neuprint_dataset_choice(
                EDGE_NEUPRINT_DATASET_FAMILY,
                EDGE_NEUPRINT_DATASET_VERSION or _default_version,
            )
        _edge_request = _phase2_ensure_named_edge_set(
            edge_set_name=EDGE_SET_NAME,
            requested_ids=loaded_ids,
            swc_dir=SWC_DIR,
            selection_mode="custom",
            selection_label=RUN_ID,
            expansion_mode=CUSTOM_EDGE_EXPANSION_MODE,
            master_csv=(MASTER_CSV if MASTER_CSV else None),
            neuprint_dataset=EDGE_NEUPRINT_DATASET,
            registry_root=(EDGES_REGISTRY_ROOT or (_phase2_default_edges_registry_root() if _phase2_default_edges_registry_root else None)),
            force_rebuild=False,
            default_weight_uS=float(DEFAULT_WEIGHT_uS),
            one_row_per_synapse=True,
            workers=int(BUILD_EDGES_WORKERS) if 'BUILD_EDGES_WORKERS' in globals() else 16,
            phase1_fallback_enabled=bool(PHASE1_FALLBACK_ENABLED),
            phase1_upsample_nm=float(PHASE1_UPSAMPLE_NM),
            phase1_min_conf=float(PHASE1_MIN_CONF),
            phase1_batch_size=int(PHASE1_BATCH_SIZE),
        )
        EDGES_PATH = str(Path(_edge_request["final_edges_path"]).expanduser().resolve())
        print("[edges] using named edge set:", _edge_request["edge_set_name"])
        print("[edges] registry root       :", _edge_request["registry_root"])
        print("[edges] dataset family      :", EDGE_NEUPRINT_DATASET_FAMILY)
        print("[edges] dataset version     :", EDGE_NEUPRINT_DATASET_VERSION)
        print("[edges] neuPrint dataset    :", _edge_request.get("neuprint_dataset", EDGE_NEUPRINT_DATASET))
        print("[edges] final edges path    :", EDGES_PATH)
        print("[edges] final network count :", len(_edge_request["final_network_ids"]))
        if CUSTOM_EDGE_EXPANSION_MODE == "immediate_motor_postsynaptic":
            selection["neuron_ids"] = list(_edge_request["final_network_ids"])
            loaded_ids = list(selection["neuron_ids"])
            missing = sorted(set(SEEDS) - set(loaded_ids))
            if missing:
                raise ValueError(
                    "Stim seeds fell out of the auto-built custom network.\n"
                    f"Loaded neuron_ids: {loaded_ids}\n"
                    f"Stim seeds:        {SEEDS}\n"
                    f"Missing from load: {missing}"
                )
            print("[edges] custom expansion mode added ids -> total loaded neurons =", len(loaded_ids))

# -------------------------------------------------------------------
# Custom-edge sanity and augmentation (custom mode)
# -------------------------------------------------------------------
# If a biologically expected pair is missing OR underrepresented in EDGES_PATH,
# patch it in from pair CSV and/or rawsyn edges.
REQUIRED_PAIRS = [(10000, 10110), (10002, 10068)]
# Minimum synapse-row targets for required pairs (after loading base edges).
REQUIRED_PAIR_MIN_ROWS = {(10000, 10110): 146, (10002, 10068): 135}
AUTO_DROP_DISCONNECTED = True
EDGES_PATH_EFFECTIVE = EDGES_PATH

# Optional chemical-only added connections (direction-aware): source->postsyn by default.
CHEM_ONLY_DIRECTION = str(globals().get("CHEM_ONLY_CONNECTION_DIRECTION", "source_to_postsyn")).strip().lower()
_chem_map_raw = globals().get("CHEM_ONLY_OUTPUTS_BY_SOURCE", None)
if _chem_map_raw is None:
    _chem_map_raw = globals().get("CHEM_ONLY_INPUTS_BY_TARGET", {}) or {}
CHEM_ONLY_MAP = _chem_map_raw if isinstance(_chem_map_raw, dict) else {}
if _phase2_build_chem_only_pairs is not None:
    _CHEM_ONLY_SPEC = _phase2_build_chem_only_pairs(CHEM_ONLY_MAP, direction=CHEM_ONLY_DIRECTION)
else:
    _CHEM_ONLY_SPEC = {"pairs": [], "added_ids": []}
CHEM_ONLY_EDGE_PAIRS = [tuple(map(int, p)) for p in (_CHEM_ONLY_SPEC.get("pairs") or [])]
CHEM_ONLY_ADDED_IDS = [int(x) for x in (_CHEM_ONLY_SPEC.get("added_ids") or [])]
if MODE == "custom" and CHEM_ONLY_EDGE_PAIRS:
    print("[chem-only] edge direction =", CHEM_ONLY_DIRECTION)
    print("[chem-only] added neuron ids =", CHEM_ONLY_ADDED_IDS)

if MODE == "custom" and EDGE_CACHE_ENABLED:
    print("[edges] using edge_cache query mode; manual overlay augmentation is bypassed (runner may fallback to EDGES_PATH if cache subset is disconnected).")

def _read_edges_table(p: Path):
    sfx = p.suffix.lower()
    if sfx == ".csv":
        return pd.read_csv(p)
    if sfx in {".parquet", ".pq"}:
        return pd.read_parquet(p)
    if sfx in {".feather", ".ftr"}:
        return pd.read_feather(p)
    return pd.read_csv(p)

def _normalize_pair_rows(df_pair: pd.DataFrame, pre: int, post: int) -> pd.DataFrame:
    if df_pair is None or df_pair.empty:
        return pd.DataFrame()

    d = pd.DataFrame(index=df_pair.index.copy())
    d["pre_id"] = int(pre)
    d["post_id"] = int(post)

    for src, dst in [
        ("pre_x", "pre_x"), ("x_pre", "pre_x"),
        ("pre_y", "pre_y"), ("y_pre", "pre_y"),
        ("pre_z", "pre_z"), ("z_pre", "pre_z"),
        ("post_x", "post_x"), ("x_post", "post_x"),
        ("post_y", "post_y"), ("y_post", "post_y"),
        ("post_z", "post_z"), ("z_post", "post_z"),
        ("pre_syn_index", "pre_syn_index"),
        ("post_syn_index", "post_syn_index"),
    ]:
        if src in df_pair.columns and dst not in d.columns:
            d[dst] = pd.to_numeric(df_pair[src], errors="coerce")

    # Ensure these overlay rows survive epsilon filtering in builders.py.
    d["pre_match_um"] = 0.0
    d["post_match_um"] = 0.0

    # Preserve timing fields if available; otherwise leave NaN to use config defaults.
    for src, dst in [
        ("weight_uS", "weight_uS"),
        ("delay_ms", "delay_ms"),
        ("syn_e_rev_mV", "syn_e_rev_mV"),
        ("e_rev_mV", "syn_e_rev_mV"),
        ("tau1_ms", "tau1_ms"),
        ("tau2_ms", "tau2_ms"),
    ]:
        if src in df_pair.columns and dst not in d.columns:
            d[dst] = pd.to_numeric(df_pair[src], errors="coerce")

    for c in ["weight_uS", "delay_ms", "syn_e_rev_mV", "tau1_ms", "tau2_ms"]:
        if c not in d.columns:
            d[c] = float("nan")

    dedup_cols = [
        c for c in [
            "pre_id", "post_id", "pre_x", "pre_y", "pre_z", "post_x", "post_y", "post_z",
            "pre_syn_index", "post_syn_index",
        ]
        if c in d.columns
    ]
    if dedup_cols:
        d = d.drop_duplicates(subset=dedup_cols, keep="first")

    return d

def _load_pair_rows_from_pairs_csv(pair_csv: Path, pre: int, post: int) -> pd.DataFrame:
    if not pair_csv.exists():
        return pd.DataFrame()
    df_pair = pd.read_csv(pair_csv)
    return _normalize_pair_rows(df_pair, pre, post)

def _load_pair_rows_from_rawsyn(search_root: Path, pre: int, post: int):
    pattern = f"edges_ego_{int(pre)}__rawsyn.csv"
    for cand in sorted(search_root.rglob(pattern)):
        try:
            df_raw = pd.read_csv(cand)
        except Exception:
            continue
        if not {"pre_id", "post_id"}.issubset(set(df_raw.columns)):
            continue
        keep = df_raw[
            (pd.to_numeric(df_raw["pre_id"], errors="coerce").fillna(-1).astype(int) == int(pre))
            & (pd.to_numeric(df_raw["post_id"], errors="coerce").fillna(-1).astype(int) == int(post))
        ].copy()
        if keep.empty:
            continue
        return _normalize_pair_rows(keep, pre, post), cand
    return pd.DataFrame(), None

if MODE == "custom" and EDGES_PATH and (not EDGE_CACHE_ENABLED):
    edge_path = Path(EDGES_PATH).expanduser().resolve()
    try:
        df_edges = _read_edges_table(edge_path)
        if not {"pre_id", "post_id"}.issubset(set(df_edges.columns)):
            raise ValueError(f"edges table missing pre_id/post_id: {edge_path}")

        loaded_set = set(int(x) for x in loaded_ids)

        _pairs = df_edges[["pre_id", "post_id"]].copy()
        _pairs["pre_id"] = pd.to_numeric(_pairs["pre_id"], errors="coerce").fillna(-1).astype(int)
        _pairs["post_id"] = pd.to_numeric(_pairs["post_id"], errors="coerce").fillna(-1).astype(int)
        pair_counts = _pairs.groupby(["pre_id", "post_id"]).size().to_dict()

        deficit_required = []
        for pre, post in REQUIRED_PAIRS:
            pre = int(pre)
            post = int(post)
            if pre not in loaded_set or post not in loaded_set:
                continue
            have = int(pair_counts.get((pre, post), 0))
            need = int(REQUIRED_PAIR_MIN_ROWS.get((pre, post), 1))
            if have < need:
                deficit_required.append((pre, post, have, need))

        added_pairs = []
        if deficit_required:
            extras = []
            swc_root = Path(SWC_DIR).expanduser().resolve()

            for pre, post, have, need in deficit_required:
                source_rows = []

                pair_csv = edge_path.parent / f"pairs_{pre}_to_{post}.csv"
                d_pairs = _load_pair_rows_from_pairs_csv(pair_csv, pre, post)

                d_raw, raw_path = _load_pair_rows_from_rawsyn(swc_root, pre, post)

                # Prefer raw synapse rows when available; use pairs_csv only as fallback.
                if not d_raw.empty:
                    source_rows.append(("rawsyn_csv", str(raw_path), d_raw))
                elif not d_pairs.empty:
                    source_rows.append(("pairs_csv", str(pair_csv), d_pairs))

                if not source_rows:
                    continue

                # Replace any existing rows for this pair so sparse placeholders don't survive.
                _pre = pd.to_numeric(df_edges["pre_id"], errors="coerce").fillna(-1).astype(int)
                _post = pd.to_numeric(df_edges["post_id"], errors="coerce").fillna(-1).astype(int)
                df_edges = df_edges[~((_pre == pre) & (_post == post))].copy()

                d_all = pd.concat([d for _, _, d in source_rows], ignore_index=True, sort=False)
                dedup_cols = [
                    c for c in [
                        "pre_id", "post_id", "pre_x", "pre_y", "pre_z", "post_x", "post_y", "post_z",
                        "pre_syn_index", "post_syn_index",
                    ]
                    if c in d_all.columns
                ]
                if dedup_cols:
                    d_all = d_all.drop_duplicates(subset=dedup_cols, keep="first")

                extras.append(d_all)
                added_pairs.append((
                    pre,
                    post,
                    have,
                    need,
                    int(len(d_all)),
                    [f"{typ}:{src}" for typ, src, _ in source_rows],
                ))

            if extras:
                df_edges = pd.concat([df_edges] + extras, ignore_index=True, sort=False)

                # Sanity-clean ids before writing overlay.
                for _idc in ["pre_id", "post_id"]:
                    df_edges[_idc] = pd.to_numeric(df_edges[_idc], errors="coerce")
                before_n = len(df_edges)
                df_edges = df_edges.dropna(subset=["pre_id", "post_id"]).copy()
                if len(df_edges) != before_n:
                    print(f"[edges] dropped malformed overlay rows: {before_n - len(df_edges)}")
                df_edges["pre_id"] = df_edges["pre_id"].astype(int)
                df_edges["post_id"] = df_edges["post_id"].astype(int)

                # Drop coordinate-less placeholder rows for required pairs when real rows exist.
                _coord_cols = [c for c in ["pre_x", "pre_y", "pre_z", "post_x", "post_y", "post_z"] if c in df_edges.columns]
                if _coord_cols:
                    _drop_n = 0
                    for _pre, _post in REQUIRED_PAIRS:
                        _pre = int(_pre); _post = int(_post)
                        _m_pair = (df_edges["pre_id"] == _pre) & (df_edges["post_id"] == _post)
                        if not _m_pair.any():
                            continue
                        _coords = df_edges.loc[_m_pair, _coord_cols]
                        _all_nan = _coords.isna().all(axis=1)
                        _valid_n = int((~_all_nan).sum())
                        if _valid_n > 0 and int(_all_nan.sum()) > 0:
                            _idx_bad = _coords.index[_all_nan]
                            _drop_n += int(len(_idx_bad))
                            df_edges = df_edges.drop(index=_idx_bad)
                    if _drop_n:
                        print(f"[edges] dropped coordinate-less placeholder rows for required pairs: {_drop_n}")

                overlay_dir = Path.cwd() / ".edge_overlays"
                overlay_dir.mkdir(parents=True, exist_ok=True)
                overlay_path = overlay_dir / f"{RUN_ID}_custom_edges_overlay.csv"
                df_edges.to_csv(overlay_path, index=False)
                EDGES_PATH_EFFECTIVE = str(overlay_path)

                summary_rows = [(a, b, have, need, add) for a, b, have, need, add, _ in added_pairs]
                print("[edges] augmented required sparse pairs via overlay:", summary_rows)
                for a, b, _, _, _, srcs in added_pairs:
                    print(f"[edges] sources for {a}->{b}: {srcs}")
                print(f"[edges] overlay path: {overlay_path}")
            else:
                print(
                    "[edges] required pairs are sparse, but no usable pair sources found:",
                    [(a, b, have, need) for a, b, have, need in deficit_required],
                )

        # Always sanitize and clean required pairs, even when no augmentation was needed.
        _wrote_clean_overlay = False
        for _idc in ["pre_id", "post_id"]:
            df_edges[_idc] = pd.to_numeric(df_edges[_idc], errors="coerce")
        _before_n = len(df_edges)
        df_edges = df_edges.dropna(subset=["pre_id", "post_id"]).copy()
        if len(df_edges) != _before_n:
            print(f"[edges] dropped malformed rows during cleanup: {_before_n - len(df_edges)}")
            _wrote_clean_overlay = True
        df_edges["pre_id"] = df_edges["pre_id"].astype(int)
        df_edges["post_id"] = df_edges["post_id"].astype(int)

        _coord_cols = [c for c in ["pre_x", "pre_y", "pre_z", "post_x", "post_y", "post_z"] if c in df_edges.columns]
        if _coord_cols:
            _drop_n = 0
            for _pre, _post in REQUIRED_PAIRS:
                _pre = int(_pre); _post = int(_post)
                _m_pair = (df_edges["pre_id"] == _pre) & (df_edges["post_id"] == _post)
                if not _m_pair.any():
                    continue
                _coords = df_edges.loc[_m_pair, _coord_cols]
                _all_nan = _coords.isna().all(axis=1)
                _valid_n = int((~_all_nan).sum())
                if _valid_n > 0 and int(_all_nan.sum()) > 0:
                    _idx_bad = _coords.index[_all_nan]
                    _drop_n += int(len(_idx_bad))
                    df_edges = df_edges.drop(index=_idx_bad)
            if _drop_n:
                print(f"[edges] dropped coordinate-less placeholder rows for required pairs: {_drop_n}")
                _wrote_clean_overlay = True

        if _wrote_clean_overlay:
            overlay_dir = Path.cwd() / ".edge_overlays"
            overlay_dir.mkdir(parents=True, exist_ok=True)
            overlay_path = overlay_dir / f"{RUN_ID}_custom_edges_overlay.csv"
            df_edges.to_csv(overlay_path, index=False)
            EDGES_PATH_EFFECTIVE = str(overlay_path)
            print(f"[edges] wrote cleaned overlay: {overlay_path}")

        # Recompute connectivity from the effective edge table.
        _pp = df_edges[["pre_id", "post_id"]].copy()
        _pp["pre_id"] = pd.to_numeric(_pp["pre_id"], errors="coerce").fillna(-1).astype(int)
        _pp["post_id"] = pd.to_numeric(_pp["post_id"], errors="coerce").fillna(-1).astype(int)
        _sub = _pp[_pp["pre_id"].isin(loaded_set) & _pp["post_id"].isin(loaded_set)]
        _indeg = _sub.groupby("post_id").size().to_dict()
        _outdeg = _sub.groupby("pre_id").size().to_dict()
        _pair_counts = _sub.groupby(["pre_id", "post_id"]).size().to_dict()
        _pair_rows = sorted(
            [(int(a), int(b), int(n)) for (a, b), n in _pair_counts.items()],
            key=lambda x: (-x[2], x[0], x[1]),
        )
        print('[connectivity] pair row counts among loaded ids:', _pair_rows)

        # Print per-node connectivity so disconnects are obvious before run.
        _conn_rows = []
        for _nid in loaded_ids:
            _nid = int(_nid)
            _conn_rows.append((
                _nid,
                int(_indeg.get(_nid, 0)),
                int(_outdeg.get(_nid, 0)),
            ))
        print('[connectivity] indeg/outdeg among loaded ids:', _conn_rows)

        if AUTO_DROP_DISCONNECTED:
            disconnected = [
                int(nid)
                for nid in loaded_ids
                if (int(nid) not in set(SEEDS)) and int(_indeg.get(int(nid), 0)) == 0
            ]
            if disconnected:
                loaded_ids = [int(nid) for nid in loaded_ids if int(nid) not in set(disconnected)]
                selection["neuron_ids"] = list(loaded_ids)
                print(
                    "[connectivity] dropped disconnected non-seed IDs "
                    f"(no incoming edges in selected subgraph): {disconnected}"
                )
    except Exception as e:
        print(f"[connectivity] check skipped: {e}")

# -------------------------------------------------------------------
# Recording policy:
#   - spikes for ALL loaded neurons (so downstream firing is captured)
#   - soma_v for stim neurons only by default (lighter)
#     set RECORD_SOMA_FOR_ALL=True for soma voltage traces from every loaded neuron
# -------------------------------------------------------------------
RECORD_SOMA_FOR_ALL = True  # set True for soma voltage traces from every loaded neuron

if MODE == "custom":
    loaded_ids = list(selection["neuron_ids"])
    RECORD_DYN = _apply_recording_policy(
        record_dict=RECORD,
        loaded_ids=loaded_ids,
        stim_ids=SEEDS,
        record_soma_for_all=RECORD_SOMA_FOR_ALL,
    )
else:
    RECORD_DYN = dict(RECORD)

# -------------------------------------------------------------------
# USER_OVERRIDES takes priority over defaults.
# -------------------------------------------------------------------
USER_OVERRIDES = {
    # Required/derived paths
    "swc_dir": str(Path(SWC_DIR).expanduser().resolve()),
    "morph_swc_dir": (str(Path(MORPH_SWC_DIR).expanduser().resolve()) if MORPH_SWC_DIR else None),
    "master_csv": str(Path(MASTER_CSV).expanduser().resolve()) if MASTER_CSV else None,
    "edges_root": str(Path(EDGES_ROOT).expanduser().resolve()) if EDGES_ROOT else None,
    "runs_root": str(Path(RUNS_ROOT).expanduser().resolve()) if RUNS_ROOT else None,

    # Selection + edges
    "selection": selection,
    "seeds": SEEDS,
    "edges_path": (str(Path(EDGES_PATH_EFFECTIVE).expanduser().resolve()) if (MODE == "custom" and EDGES_PATH_EFFECTIVE) else None),
    "edge_cache": {
        "enabled": bool(EDGE_CACHE_ENABLED and MODE == "custom"),
        "db_path": (str(Path(EDGE_CACHE_DB_PATH).expanduser().resolve()) if EDGE_CACHE_DB_PATH else None),
        "build_if_missing": bool(EDGE_CACHE_BUILD_IF_MISSING),
        "force_rebuild": bool(EDGE_CACHE_FORCE_REBUILD),
        "build_mode": str(EDGE_CACHE_BUILD_MODE),
        "source_paths": (
            [str(Path(p).expanduser().resolve()) for p in EDGE_CACHE_SOURCE_PATHS]
            if EDGE_CACHE_SOURCE_PATHS else None
        ),
        "query": {
            "mode": str(EDGE_CACHE_QUERY_MODE),
            "include_loaded": bool(EDGE_CACHE_INCLUDE_LOADED),
            "max_nodes": EDGE_CACHE_MAX_NODES,
            "max_rows": EDGE_CACHE_MAX_ROWS,
        },
    },

    # Run identity + clock
    "run_id": RUN_ID,
    "run_notes": RUN_NOTES,
    "tstop_ms": float(TSTOP_MS),
    "dt_ms": float(DT_MS),
    "celsius_C": float(CELSIUS_C),

    # Category 1: runtime
    "progress": bool(PROGRESS),
    "use_tqdm": bool(USE_TQDM),
    "progress_chunk_ms": float(PROGRESS_CHUNK_MS),
    "threads": THREADS,
    "enable_coreneuron": bool(ENABLE_CORENEURON),
    "coreneuron_gpu": bool(CORENEURON_GPU),
    "coreneuron_nthread": CORENEURON_NTHREAD,
    "cvode": dict(CVODE),

    # Category 6: IO
    "io_workers": IO_WORKERS,
    "io_parallel": IO_PARALLEL,

    # Category 2: wiring robustness
    "wire_force_soma": bool(WIRE_FORCE_SOMA),
    "coalesce_syns": bool(COALESCE_SYNS),
    "coalesce_mode": str(COALESCE_MODE),
    "min_weight_uS": float(MIN_WEIGHT_uS) if MIN_WEIGHT_uS is not None else None,
    "coalesce_guard_w_med_uS": COALESCE_GUARD_W_MED_uS,
    "coalesce_guard_drop": COALESCE_GUARD_DROP,
    "use_geom_delay": bool(USE_GEOM_DELAY),

    # Category 3: timing defaults + epsilon
    "default_weight_uS": DEFAULT_WEIGHT_uS,
    "default_delay_ms": DEFAULT_DELAY_MS,
    "global_timing": {
        "global_weight_scale": float(GLOBAL_WEIGHT_SCALE),
        "base_release_delay_ms": float(GLOBAL_BASE_RELEASE_DELAY_MS),
        "vel_um_per_ms": float(GLOBAL_VEL_UM_PER_MS),
    },
    "syn_tau1_ms": SYN_TAU1_MS,
    "syn_tau2_ms": SYN_TAU2_MS,
    "syn_e_rev_mV": SYN_E_REV_MV,
    "epsilon_um": EPSILON_UM,

    # Category 4: AIS mapping
    "ais_min_dist_um": AIS_MIN_DIST_UM,
    "ais_min_xloc": AIS_MIN_XLOC,

    # Category 5: biophys
    "v_rest_mV": float(V_REST_MV),
    "v_init_mV": float(V_INIT_MV),
    "ena_mV": E_NA_MV,
    "ek_mV": E_K_MV,
    "el_mV": E_L_MV,
    "passive_global": dict(PASSIVE_GLOBAL),
    "hh_global": (dict(HH_GLOBAL) if isinstance(HH_GLOBAL, dict) else None),

    # Master-compatible explicit overrides (optional)
    "pre_soma_hh": PRE_SOMA_HH,
    "pre_branch_hh": PRE_BRANCH_HH,
    "post_soma_hh": POST_SOMA_HH,
    "post_branch_hh": POST_BRANCH_HH,
    "post_active": bool(POST_ACTIVE),

    # Stim
    "stim": {"iclamp": dict(ICLAMP), "neg_pulse": dict(NEG_PULSE), "pulse_train": (dict(PULSE_TRAIN) if "PULSE_TRAIN" in globals() else None)},
    "gap": dict(GAP),

    # Record + silence
    "record": dict(RECORD_DYN),
    "silence": dict(SILENCE),
}

# Auto performance preset for large 1-hop cache runs
if (
    MODE == "custom"
    and bool(EDGE_CACHE_ENABLED)
    and str(EDGE_CACHE_QUERY_MODE).strip().lower() == "seed_io_1hop"
    and bool(FAST_1HOP_MODE)
):
    USER_OVERRIDES["coalesce_syns"] = True
    USER_OVERRIDES["coalesce_mode"] = str(FAST_1HOP_COALESCE_MODE)
    USER_OVERRIDES["progress_chunk_ms"] = float(FAST_1HOP_PROGRESS_CHUNK_MS)
    if bool(FAST_1HOP_DISABLE_GEOM_DELAY):
        USER_OVERRIDES["use_geom_delay"] = False
    if bool(FAST_1HOP_DISABLE_CVODE):
        USER_OVERRIDES["cvode"] = {"enabled": False}
    if bool(FAST_1HOP_FORCE_CORENEURON):
        USER_OVERRIDES["enable_coreneuron"] = True
        if bool(FAST_1HOP_FORCE_GPU):
            USER_OVERRIDES["coreneuron_gpu"] = True
    print(
        "[perf] FAST_1HOP_MODE applied: "
        f"coalesce_mode={USER_OVERRIDES.get('coalesce_mode')} "
        f"use_geom_delay={USER_OVERRIDES.get('use_geom_delay')} "
        f"core={USER_OVERRIDES.get('enable_coreneuron')} gpu={USER_OVERRIDES.get('coreneuron_gpu')} "
        f"cvode.enabled={bool((USER_OVERRIDES.get('cvode') or {}).get('enabled', False))} "
        f"progress_chunk_ms={USER_OVERRIDES.get('progress_chunk_ms')}"
    )

USER_OVERRIDES = _strip_none(USER_OVERRIDES)
if USER_OVERRIDES.get("enable_coreneuron", False) and bool((USER_OVERRIDES.get("cvode", {}) or {}).get("enabled", False)):
    print("[cfg] CoreNEURON and CVODE both requested; forcing cvode.enabled=False.")
    USER_OVERRIDES["cvode"] = {"enabled": False}

# -------------------------------------------------------------------
# Debug prints for run-by-run behavior checks.
# -------------------------------------------------------------------
print("\n[CONFIG PREVIEW]")
print("  MODE            =", MODE)
print("  selection       =", selection)
print("  loaded neuron_ids =", selection.get("neuron_ids") if MODE == "custom" else None)
print("  stim seeds      =", SEEDS)
print("  edges_path      =", USER_OVERRIDES.get("edges_path"))
print("  edge_cache.en   =", bool((USER_OVERRIDES.get("edge_cache", {}) or {}).get("enabled", False)))
print("  edge_cache.mode =", ((USER_OVERRIDES.get("edge_cache", {}) or {}).get("query", {}) or {}).get("mode", None))
print("  morph_swc_dir   =", USER_OVERRIDES.get("morph_swc_dir"))
print("  record keys     =", list(USER_OVERRIDES.get("record", {}).keys()))
print("  record.spikes   =", USER_OVERRIDES.get("record", {}).get("spikes", None))
print("  record.soma_v   =", USER_OVERRIDES.get("record", {}).get("soma_v", None))
print("  gap.enabled     =", USER_OVERRIDES.get("gap", {}).get("enabled", False))
print("  gap.pairs       =", len(USER_OVERRIDES.get("gap", {}).get("pairs", [])))
print("  tstop_ms / dt_ms=", USER_OVERRIDES.get("tstop_ms"), "/", USER_OVERRIDES.get("dt_ms"))
print("  celsius_C       =", USER_OVERRIDES.get("celsius_C"))
print("  backend/core    =", "coreneuron" if USER_OVERRIDES.get("enable_coreneuron", False) else "neuron")
_cv = USER_OVERRIDES.get("cvode", {}) or {}
print("  cvode.enabled   =", bool(_cv.get("enabled", False)))
print("  cvode.atol/rtol =", _cv.get("atol", None), "/", _cv.get("rtol", None))
print("  cvode.maxstep   =", _cv.get("maxstep_ms", None))
print("  stim.pulse_train=", (USER_OVERRIDES.get("stim", {}) or {}).get("pulse_train", None))
print("")
if MODE == "custom" and CHEM_ONLY_EDGE_PAIRS:
    print("  chem-only.dir    =", CHEM_ONLY_DIRECTION)
    print("  chem-only.added  =", CHEM_ONLY_ADDED_IDS)
    print("")

# -------------------------------------------------------------------
# Build final CONFIG and run
# -------------------------------------------------------------------
CONFIG = build_config(USER_OVERRIDES)
if _phase2_apply_chem_only_workflow_to_cfg is not None and MODE == "custom" and CHEM_ONLY_EDGE_PAIRS:
    CONFIG, _chem_shared_info = _phase2_apply_chem_only_workflow_to_cfg(
        CONFIG,
        scenario_label=str(RUN_ID),
        chem_only_map=CHEM_ONLY_MAP,
        chem_only_direction=CHEM_ONLY_DIRECTION,
        record_soma_for_all=bool(RECORD_SOMA_FOR_ALL),
        force_overlay_for_missing_pairs=True,
        force_min_rows_per_pair=int(globals().get("CHEM_ONLY_FORCE_MIN_ROWS_PER_PAIR", 128)),
        force_weight_scale=float(globals().get("CHEM_ONLY_FORCE_WEIGHT_SCALE", 1.0)),
        force_delay_ms=globals().get("CHEM_ONLY_FORCE_DELAY_MS", None),
        force_erev_mV=globals().get("CHEM_ONLY_FORCE_EREV_MV", None),
        force_tau1_ms=globals().get("CHEM_ONLY_FORCE_TAU1_MS", None),
        force_tau2_ms=globals().get("CHEM_ONLY_FORCE_TAU2_MS", None),
        force_wire_soma=bool(globals().get("CHEM_ONLY_FORCE_WIRE_SOMA", False)),
        base_edges_path=globals().get("EDGES_PATH_EFFECTIVE", None),
        swc_root=(globals().get("MORPH_SWC_DIR", None) or globals().get("SWC_DIR", None)),
        verbose=True,
    )
else:
    _chem_shared_info = {"applied": False, "note": "shared workflow no-op"}

print("CONFIG built.")
print("  swc_dir   =", CONFIG.get("swc_dir"))
print("  edges_root=", CONFIG.get("edges_root"))
print("  runs_root =", CONFIG.get("runs_root"))
print("  master_csv=", CONFIG.get("master_csv"))
print("  selection =", CONFIG.get("selection"))
print("  run_id    =", CONFIG.get("run_id"))
print("  seeds     =", CONFIG.get("seeds"))
print("  record    =", CONFIG.get("record"))
if MODE == "custom" and CHEM_ONLY_EDGE_PAIRS:
    print("  chem-only.shared=", _chem_shared_info)

out_dir = run_walking_simulation(CONFIG)
print("Results saved to:", out_dir)



In [ ]:
import json
from pathlib import Path
cfg = json.loads((Path(CONFIG["runs_root"]) / RUN_ID / "config.json").read_text())
print(cfg.get("morph_swc_dir"))


In [ ]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path(out_dir) / "records.csv")
mx = df.drop(columns=["t_ms"]).max()
print(mx.sort_values(ascending=False).head(20))


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
OUT_DIR = Path(out_dir).expanduser().resolve()
PLOT_DIR = OUT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("OUT_DIR =", OUT_DIR)
print("PLOT_DIR=", PLOT_DIR)

cfg_path = OUT_DIR / "config.json"
rec_path = OUT_DIR / "records.csv"

if not cfg_path.exists():
    raise FileNotFoundError(f"Missing: {cfg_path}")
if not rec_path.exists():
    raise FileNotFoundError(f"Missing: {rec_path}")

CFG = json.loads(cfg_path.read_text(encoding="utf-8"))
DF  = pd.read_csv(rec_path)

# ------------------------------------------------------------
# Determine which neuron IDs have soma traces in records.csv
# records.csv columns look like:
#   t_ms, 10000_soma_v, 10002_soma_v, ...
# ------------------------------------------------------------
time_col_candidates = ["t_ms", "time_ms", "t"]
tcol = None
for c in time_col_candidates:
    if c in DF.columns:
        tcol = c
        break
if tcol is None:
    raise ValueError(f"records.csv missing time column. Found: {list(DF.columns)[:30]}")

t = DF[tcol].to_numpy(dtype=float)

soma_cols = []
for c in DF.columns:
    if re.fullmatch(r"\d+_soma_v", str(c)):
        soma_cols.append(c)

if not soma_cols:
    # fall back: some builds use "{nid}__soma_v" or "{nid}_soma_v__0"
    for c in DF.columns:
        if re.search(r"_soma_v", str(c)):
            soma_cols.append(c)

if not soma_cols:
    raise RuntimeError(
        "No soma voltage columns found in records.csv.\n"
        "Expected columns like '10000_soma_v'.\n"
        f"Top columns: {list(DF.columns)[:40]}"
    )

# Parse IDs from columns
def _parse_nid(col: str) -> int | None:
    m = re.match(r"^(\d+)", str(col))
    return int(m.group(1)) if m else None

ids_from_records = []
for c in soma_cols:
    nid = _parse_nid(c)
    if nid is not None and nid not in ids_from_records:
        ids_from_records.append(nid)

# Prefer selection neuron_ids ordering if present (custom mode)
selection = CFG.get("selection", {}) or {}
mode = selection.get("mode", None)

if mode == "custom" and selection.get("neuron_ids"):
    ordered_ids = [int(x) for x in selection["neuron_ids"]]
    # keep only those that actually have soma traces
    ordered_ids = [nid for nid in ordered_ids if nid in ids_from_records]
else:
    ordered_ids = list(ids_from_records)

# Append any extra recorded soma traces not present in config selection order.
# This captures added chemical-only neurons when an older/stale selection list in
# config.json still has only the original circuit subset.
extra_recorded_ids = [nid for nid in ids_from_records if nid not in ordered_ids]
if extra_recorded_ids:
    chem_added_order = []
    for _chem_name in ("CHEM_ONLY_OUTPUTS_BY_SOURCE", "CHEM_ONLY_INPUTS_BY_TARGET"):
        _chem_map = globals().get(_chem_name, None)
        if isinstance(_chem_map, dict) and _chem_map:
            _seen_added = set()
            for _vals in _chem_map.values():
                for _nid in (_vals or []):
                    _ni = int(_nid)
                    if _ni in _seen_added:
                        continue
                    _seen_added.add(_ni)
                    chem_added_order.append(_ni)
            break

    _to_append = [nid for nid in chem_added_order if nid in extra_recorded_ids]
    _to_append += [nid for nid in extra_recorded_ids if nid not in set(_to_append)]
    ordered_ids = list(ordered_ids) + _to_append
    print("[rec] appended extra recorded soma ids =", _to_append)

seeds = [int(x) for x in (CFG.get("seeds") or [])]

print("[cfg] mode  =", mode)
print("[cfg] seeds =", seeds)
print("[rec] soma traces for ids =", ordered_ids)

# Guardrail: catch broken recordings early (all-NaN after first sample)
finite_counts = {}
for nid in ordered_ids:
    exact = f"{nid}_soma_v"
    if exact in DF.columns:
        col = exact
    else:
        candidates = [c for c in soma_cols if str(c).startswith(f"{nid}") and ("soma_v" in str(c))]
        if not candidates:
            continue
        col = candidates[0]
    finite_counts[int(nid)] = int(np.isfinite(pd.to_numeric(DF[col], errors="coerce")).sum())

if finite_counts and max(finite_counts.values()) <= 1:
    raise RuntimeError(
        "records.csv has only an initial sample and then NaNs for soma traces.\n"
        "This indicates stale runtime modules in kernel memory.\n"
        "Fix: rerun the bootstrap cell (module reload), then rerun the simulation cell.\n"
        f"Finite counts by nid: {finite_counts}"
    )

# ------------------------------------------------------------
# Helper: threshold-crossing spike times from soma V (optional markers)
# (This is NOT a raster; markers are plotted on the trace itself.)
# Threshold matches the NetCon threshold=0.0 mV demo setting.
# ------------------------------------------------------------
def _spike_times_from_trace(t_ms: np.ndarray, v_mV: np.ndarray, thresh_mV: float = 0.0) -> np.ndarray:
    v = np.asarray(v_mV, dtype=float)
    tt = np.asarray(t_ms, dtype=float)
    if v.size != tt.size or v.size < 2:
        return np.array([], dtype=float)
    # upward crossings
    cross = np.where((v[:-1] < thresh_mV) & (v[1:] >= thresh_mV))[0]
    if cross.size == 0:
        return np.array([], dtype=float)
    # linear interpolation for better timing
    t0 = tt[cross]
    t1 = tt[cross + 1]
    v0 = v[cross]
    v1 = v[cross + 1]
    denom = np.where(np.abs(v1 - v0) < 1e-12, 1e-12, (v1 - v0))
    frac = (thresh_mV - v0) / denom
    return t0 + frac * (t1 - t0)

# ------------------------------------------------------------
# Latency report for key pairs (first-spike latency, ms)
# ------------------------------------------------------------
def _load_spike_times(out_dir: Path) -> dict[int, np.ndarray]:
    spike_map: dict[int, np.ndarray] = {}
    for p in [out_dir / "spike_times.csv", out_dir / "spikes.csv"]:
        if not p.exists():
            continue
        S = pd.read_csv(p)
        nid_col = "neuron_id" if "neuron_id" in S.columns else ("nid" if "nid" in S.columns else None)
        t_col = "spike_time_ms" if "spike_time_ms" in S.columns else ("t_ms" if "t_ms" in S.columns else None)
        if nid_col is None or t_col is None:
            continue
        S = S[[nid_col, t_col]].copy()
        S[nid_col] = pd.to_numeric(S[nid_col], errors="coerce")
        S[t_col] = pd.to_numeric(S[t_col], errors="coerce")
        S = S.dropna()
        if S.empty:
            continue
        for nid, grp in S.groupby(nid_col):
            spike_map[int(nid)] = np.sort(grp[t_col].to_numpy(dtype=float))
        if spike_map:
            return spike_map
    return spike_map

def _latency_ms(spike_map: dict[int, np.ndarray], pre: int, post: int) -> float | None:
    a = spike_map.get(int(pre), np.array([], dtype=float))
    b = spike_map.get(int(post), np.array([], dtype=float))
    if a.size == 0 or b.size == 0:
        return None
    t_pre = float(a[0])
    t_post_candidates = b[b >= t_pre]
    if t_post_candidates.size == 0:
        return None
    return float(t_post_candidates[0] - t_pre)

spike_map = _load_spike_times(OUT_DIR)

# Fallback to threshold crossings from traces if no spike CSV was usable.
if not spike_map:
    for nid in ordered_ids:
        exact = f"{nid}_soma_v"
        if exact in DF.columns:
            col = exact
        else:
            candidates = [c for c in soma_cols if str(c).startswith(f"{nid}") and ("soma_v" in str(c))]
            if not candidates:
                continue
            col = candidates[0]
        spike_map[int(nid)] = _spike_times_from_trace(t, DF[col].to_numpy(dtype=float), thresh_mV=0.0)

for pre, post in [(10000, 10110), (10002, 10068)]:
    lat = _latency_ms(spike_map, pre, post)
    msg = f"{lat:.3f} ms" if lat is not None else "N/A (missing spike)"
    print(f"[latency] {pre}->{post}: {msg}")

# ------------------------------------------------------------
# 1) Reproduce "Seeds + downstream network (multi-neuron demo)"
#    (Same style as Master-Copy2: single axes, legend outside right)
# ------------------------------------------------------------
FIG1 = plt.figure(figsize=(9, 5))
ax = plt.gca()

for nid in ordered_ids:
    # find the matching column for this nid
    # prefer exact "{nid}_soma_v", else first that startswith nid and contains soma_v
    exact = f"{nid}_soma_v"
    if exact in DF.columns:
        col = exact
    else:
        candidates = [c for c in soma_cols if str(c).startswith(f"{nid}") and ("soma_v" in str(c))]
        if not candidates:
            continue
        col = candidates[0]

    v = DF[col].to_numpy(dtype=float)
    ax.plot(t, v, label=f"{nid} soma")

    # spike markers on the trace (optional)
    st = _spike_times_from_trace(t, v, thresh_mV=0.0)
    if st.size:
        # place markers at the interpolated spike times using the nearest index for y
        idx = np.clip(np.searchsorted(t, st), 0, len(t) - 1)
        ax.scatter(st, v[idx], s=12)

ax.set_title("Seeds + downstream network (multi-neuron demo)")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("V (mV)")
ax.grid(True)
# ---- Dynamic Y scaling across ALL plotted somas ----
all_v = []

for nid in ordered_ids:
    exact = f"{nid}_soma_v"
    if exact in DF.columns:
        col = exact
    else:
        candidates = [c for c in soma_cols if str(c).startswith(f"{nid}") and ("soma_v" in str(c))]
        if not candidates:
            continue
        col = candidates[0]

    all_v.append(DF[col].to_numpy(dtype=float))

if all_v:
    vmin = min(np.nanmin(v) for v in all_v)
    vmax = max(np.nanmax(v) for v in all_v)

    pad = 0.05 * (vmax - vmin if vmax > vmin else 1.0)
    ax.set_ylim(vmin - pad, vmax + pad)


# legend outside
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout(rect=[0, 0, 0.85, 1])

out_png1 = PLOT_DIR / "seeds_downstream_somas.pdf"
FIG1.savefig(out_png1, bbox_inches="tight")
plt.show()
plt.close(FIG1)
print("[saved]", out_png1)

# ------------------------------------------------------------
# 2) Reproduce "AIS visualization (STRICT)"
#    Uses the same strict visualizer already included in this repo
# ------------------------------------------------------------
from digifly.phase2.neuron_build.viz_ais import fix_and_visualize_soma_ais

# IMPORTANT:
# viz_ais saves next to cfg["edges_csv"]. Save it into this run folder.
# Provide a patched cfg pointing edges_csv to a dummy file inside OUT_DIR.
CFG_VIZ = dict(CFG)
CFG_VIZ["edges_csv"] = str((OUT_DIR / "_edges_for_viz.csv").resolve())

# Choose IDs to visualize:
# use the same ordered_ids (ones that actually have somas), else fall back to selection neuron_ids
AIS_IDS = list(ordered_ids)
if not AIS_IDS:
    if mode == "custom" and selection.get("neuron_ids"):
        AIS_IDS = [int(x) for x in selection["neuron_ids"]]
    else:
        AIS_IDS = [int(x) for x in (CFG.get("seeds") or [])]

print("[ais] visualize ids =", AIS_IDS)

fix_and_visualize_soma_ais(
    AIS_IDS,
    cfg=CFG_VIZ,
    enforce=True,
    tol_um=2.0,
    save_name=str((PLOT_DIR / "ais_visualization_strict.pdf").resolve()),
    report_csv=str((PLOT_DIR / "_ais_visualization_report.csv").resolve()),
    title="AIS visualization (STRICT)",
)

print("Done. Plots saved in:", PLOT_DIR)


## Diagnostic

In [ ]:
# # Optional: morphology-reduction validity check (baseline vs reduced runs)
# # Set REDUCE_VALIDATE=True after TWO runs are available to compare.
# REDUCE_VALIDATE = True

# # Baseline run should be from original SWCs (MORPH_SWC_DIR=None)
# BASELINE_RUN_ID = "run_debug_full_baseline"

# # Reduced run should be from reduced SWCs (MORPH_SWC_DIR pointing at export_swc_reduced/...)
# REDUCED_RUN_ID = "run_debug_full_reduced"

# # Neurons and pathway latencies to compare
# VALIDATE_TRACK_IDS = [10000, 10002, 10068, 10110, 11446, 11654]
# VALIDATE_LATENCY_PAIRS = [(10000, 10110), (10002, 10068)]

# # Suggested acceptance thresholds (edit as needed)
# VALIDATE_THRESHOLDS = {
#     "first_spike_ms": 0.20,
#     "latency_ms": 0.20,
#     "peak_mV": 8.0,
#     "rmse_mV": 5.0,
# }

# import numpy as np
# import pandas as pd
# from pathlib import Path
# import re

# def _load_records(out_dir: Path) -> pd.DataFrame:
#     p = out_dir / "records.csv"
#     if not p.exists():
#         raise FileNotFoundError(f"Missing records.csv: {p}")
#     return pd.read_csv(p)

# def _load_spike_times(out_dir: Path) -> pd.DataFrame:
#     p_master = out_dir / "spike_times.csv"
#     p_legacy = out_dir / "spikes.csv"
#     if p_master.exists():
#         df = pd.read_csv(p_master)
#         if {"neuron_id", "spike_time_ms"}.issubset(df.columns):
#             return df[["neuron_id", "spike_time_ms"]].copy()
#     if p_legacy.exists():
#         df = pd.read_csv(p_legacy)
#         if {"nid", "t_ms"}.issubset(df.columns):
#             return df.rename(columns={"nid": "neuron_id", "t_ms": "spike_time_ms"})[["neuron_id", "spike_time_ms"]].copy()
#     return pd.DataFrame(columns=["neuron_id", "spike_time_ms"])

# def _ensure_records_present(run_dir: Path, label: str):
#     rec = run_dir / "records.csv"
#     if rec.exists():
#         return
#     parent = run_dir.parent
#     existing = sorted([p.name for p in parent.iterdir() if p.is_dir()]) if parent.exists() else []
#     raise FileNotFoundError(
#         f"[validate] missing {label} records.csv: {rec}\\n"
#         f"[validate] existing run folders under {parent}: {existing}\\n"
#         "Create runs first:\\n"
#         "  baseline: MORPH_SWC_DIR=None, RUN_ID='run_debug_full_baseline'\\n"
#         "  reduced : MORPH_SWC_DIR='/.../export_swc_reduced/v1', RUN_ID='run_debug_full_reduced'"
#     )

# def _soma_col(df: pd.DataFrame, nid: int):
#     exact = f"{int(nid)}_soma_v"
#     if exact in df.columns:
#         return exact
#     pat = re.compile(rf"^{int(nid)}_soma_v(__\\d+)?$")
#     for c in df.columns:
#         if pat.match(c):
#             return c
#     return None

# def _first_spike_from_spike_df(df_sp: pd.DataFrame, nid: int):
#     if df_sp.empty:
#         return np.nan
#     sub = df_sp.loc[pd.to_numeric(df_sp["neuron_id"], errors="coerce") == int(nid), "spike_time_ms"]
#     if sub.empty:
#         return np.nan
#     arr = pd.to_numeric(sub, errors="coerce").dropna().to_numpy(float)
#     return float(np.min(arr)) if arr.size else np.nan

# def _trace_rmse(base_t, base_v, test_t, test_v):
#     if len(base_t) < 2 or len(test_t) < 2:
#         return np.nan
#     vb = pd.to_numeric(pd.Series(base_v), errors="coerce").to_numpy(float)
#     vt = pd.to_numeric(pd.Series(test_v), errors="coerce").to_numpy(float)
#     tb = pd.to_numeric(pd.Series(base_t), errors="coerce").to_numpy(float)
#     tt = pd.to_numeric(pd.Series(test_t), errors="coerce").to_numpy(float)
#     okb = np.isfinite(tb) & np.isfinite(vb)
#     okt = np.isfinite(tt) & np.isfinite(vt)
#     if okb.sum() < 2 or okt.sum() < 2:
#         return np.nan
#     vb_i = np.interp(tb[okb], tt[okt], vt[okt])
#     d = vb_i - vb[okb]
#     return float(np.sqrt(np.mean(d * d)))

# if REDUCE_VALIDATE:
#     if not BASELINE_RUN_ID:
#         raise ValueError("Set BASELINE_RUN_ID before running reduction validation.")

#     runs_root = Path(CONFIG.get("runs_root", Path(SWC_DIR).expanduser().resolve() / "hemi_runs")).expanduser().resolve()
#     base_dir = runs_root / str(BASELINE_RUN_ID)
#     red_dir = runs_root / str(REDUCED_RUN_ID)
#     print("[validate] baseline:", base_dir)
#     print("[validate] reduced :", red_dir)

#     _ensure_records_present(base_dir, "baseline")
#     _ensure_records_present(red_dir, "reduced")

#     base_rec = _load_records(base_dir)
#     red_rec = _load_records(red_dir)
#     base_sp = _load_spike_times(base_dir)
#     red_sp = _load_spike_times(red_dir)

#     base_t = pd.to_numeric(base_rec.get("t_ms"), errors="coerce").to_numpy(float)
#     red_t = pd.to_numeric(red_rec.get("t_ms"), errors="coerce").to_numpy(float)

#     rows = []
#     for nid in VALIDATE_TRACK_IDS:
#         cb = _soma_col(base_rec, int(nid))
#         cr = _soma_col(red_rec, int(nid))
#         if not cb or not cr:
#             rows.append({"nid": int(nid), "status": "missing_trace", "base_col": cb, "reduced_col": cr})
#             continue
#         vb = pd.to_numeric(base_rec[cb], errors="coerce").to_numpy(float)
#         vr = pd.to_numeric(red_rec[cr], errors="coerce").to_numpy(float)
#         peak_b = float(np.nanmax(vb)) if np.isfinite(vb).any() else np.nan
#         peak_r = float(np.nanmax(vr)) if np.isfinite(vr).any() else np.nan
#         fs_b = _first_spike_from_spike_df(base_sp, int(nid))
#         fs_r = _first_spike_from_spike_df(red_sp, int(nid))
#         rows.append(
#             {
#                 "nid": int(nid),
#                 "status": "ok",
#                 "peak_base_mV": peak_b,
#                 "peak_reduced_mV": peak_r,
#                 "peak_abs_diff_mV": float(abs(peak_r - peak_b)) if np.isfinite(peak_b) and np.isfinite(peak_r) else np.nan,
#                 "first_spike_base_ms": fs_b,
#                 "first_spike_reduced_ms": fs_r,
#                 "first_spike_abs_diff_ms": float(abs(fs_r - fs_b)) if np.isfinite(fs_b) and np.isfinite(fs_r) else np.nan,
#                 "trace_rmse_mV": _trace_rmse(base_t, vb, red_t, vr),
#             }
#         )

#     df_cmp = pd.DataFrame(rows)
#     if not df_cmp.empty:
#         print("\n[validate] neuron-level comparison")
#         display(df_cmp)

#     lat_rows = []
#     for pre, post in VALIDATE_LATENCY_PAIRS:
#         b_pre = _first_spike_from_spike_df(base_sp, int(pre))
#         b_post = _first_spike_from_spike_df(base_sp, int(post))
#         r_pre = _first_spike_from_spike_df(red_sp, int(pre))
#         r_post = _first_spike_from_spike_df(red_sp, int(post))
#         lat_b = float(b_post - b_pre) if np.isfinite(b_pre) and np.isfinite(b_post) else np.nan
#         lat_r = float(r_post - r_pre) if np.isfinite(r_pre) and np.isfinite(r_post) else np.nan
#         lat_rows.append(
#             {
#                 "pair": f"{int(pre)}->{int(post)}",
#                 "latency_base_ms": lat_b,
#                 "latency_reduced_ms": lat_r,
#                 "latency_abs_diff_ms": float(abs(lat_r - lat_b)) if np.isfinite(lat_b) and np.isfinite(lat_r) else np.nan,
#             }
#         )
#     df_lat = pd.DataFrame(lat_rows)
#     if not df_lat.empty:
#         print("\n[validate] pathway latency comparison")
#         display(df_lat)

#     th = dict(VALIDATE_THRESHOLDS)
#     fail_spike = 0
#     fail_peak = 0
#     fail_rmse = 0
#     fail_lat = 0
#     if not df_cmp.empty:
#         fail_spike = int((pd.to_numeric(df_cmp.get("first_spike_abs_diff_ms"), errors="coerce") > float(th["first_spike_ms"])).fillna(False).sum())
#         fail_peak = int((pd.to_numeric(df_cmp.get("peak_abs_diff_mV"), errors="coerce") > float(th["peak_mV"])).fillna(False).sum())
#         fail_rmse = int((pd.to_numeric(df_cmp.get("trace_rmse_mV"), errors="coerce") > float(th["rmse_mV"])).fillna(False).sum())
#     if not df_lat.empty:
#         fail_lat = int((pd.to_numeric(df_lat.get("latency_abs_diff_ms"), errors="coerce") > float(th["latency_ms"])).fillna(False).sum())

#     print("\n[validate] thresholds:", th)
#     print(f"[validate] fails: first_spike={fail_spike} peak={fail_peak} rmse={fail_rmse} latency={fail_lat}")
#     if (fail_spike + fail_peak + fail_rmse + fail_lat) == 0:
#         print("[validate] PASS")
#     else:
#         print("[validate] REVIEW NEEDED")
# else:
#     print("[validate] skipped (REDUCE_VALIDATE=False)")


In [ ]:
# # ======================= DROP-IN DIAGNOSTICS (FULL REPLACEMENT) =======================
# # Paste this whole cell as-is into run_simulation.ipynb (after imports, before running).
# # Requires: from neuron import h already available via NEURON installation.

# from __future__ import annotations

# from pathlib import Path
# import numpy as np
# from neuron import h

# # ------------------------------------------------------------
# # Core helpers
# # ------------------------------------------------------------
# def _has_ref(seg, name: str) -> bool:
#     return hasattr(seg, f"_ref_{name}")

# def _record_ref(vecs: dict, key: str, ref):
#     v = h.Vector()
#     v.record(ref)
#     vecs[key] = v

# def _seg_label(sec, x):
#     try:
#         return f"{sec.name()}@{float(x):.3f}"
#     except Exception:
#         return "sec@x"

# def _get_site_seg(cell, site: str):
#     """
#     site:
#       - "soma" -> cell.soma_sec(0.5)
#       - "ais"  -> cell.axon_ais_site() -> (sec, x)
#     """
#     site = site.lower().strip()
#     if site == "soma":
#         sec = cell.soma_sec
#         x = 0.5
#         return sec, x, sec(x)
#     if site == "ais":
#         sec, x = cell.axon_ais_site()
#         return sec, x, sec(x)
#     raise ValueError("site must be 'soma' or 'ais'")

# # ------------------------------------------------------------
# # Main: attach diagnostics
# # ------------------------------------------------------------
# def attach_channel_current_diagnostics(
#     net,
#     neuron_ids,
#     sites=("soma", "ais"),
#     out_dir: str | Path | None = None,
#     prefix="diag",
#     enable_imem=True,
#     include_gating=True,
#     include_hh_currents=True,
#     include_pas_current=True,
#     include_syn_current=True,
#     include_iclamp_current=True,
# ):
#     """
#     Attaches NEURON Vector recordings for:
#       - time: h.t
#       - v at soma and/or AIS
#       - HH currents: ina, ik, il (if hh present at that segment)
#       - passive current: i_pas (if pas present)
#       - total membrane current: i_membrane_ (if cvode.use_fast_imem enabled)
#       - HH gating: m, h, n (if present)
#       - synaptic currents: syn.i (records all syn objects found on net)
#       - IClamp currents: iclamp.i (records all clamp objects found on net)

#     NOTE: Must be attached BEFORE net.run().
#     Returns dict holding h.Vectors.
#     """
#     neuron_ids = [int(x) for x in neuron_ids]

#     diag = {
#         "meta": {
#             "neuron_ids": neuron_ids,
#             "sites": list(sites),
#             "prefix": str(prefix),
#         },
#         "vecs": {},       # key -> h.Vector
#         "syn_vecs": {},   # key -> h.Vector
#         "iclamp_vecs": {},# key -> h.Vector
#     }

#     # Enable fast membrane current
#     if enable_imem:
#         try:
#             h.cvode.use_fast_imem(1)
#         except Exception as e:
#             print("[diag] WARNING: could not enable fast i_membrane_:", e)

#     # Record time (ms)
#     _record_ref(diag["vecs"], "t_ms", h._ref_t)

#     hh_currents = ["ina", "ik", "il"]   # hh RANGE variables
#     hh_gating   = ["m", "h", "n"]       # hh gating RANGE variables
#     pas_vars    = ["i_pas"]             # pas RANGE variable
#     imem_vars   = ["i_membrane_"]       # computed membrane current

#     # Per-cell, per-site recordings
#     for nid in neuron_ids:
#         cell = net.ensure_cell(int(nid))

#         for site in sites:
#             sec, x, seg = _get_site_seg(cell, site)
#             tag = f"{nid}:{site}:{_seg_label(sec, x)}"

#             # Voltage
#             if _has_ref(seg, "v"):
#                 _record_ref(diag["vecs"], f"{tag}:v_mV", seg._ref_v)

#             # Total membrane current
#             if enable_imem:
#                 for var in imem_vars:
#                     if _has_ref(seg, var):
#                         _record_ref(diag["vecs"], f"{tag}:{var}", getattr(seg, f"_ref_{var}"))

#             def _record_all_current_like(diag_vecs, tag, seg):
#                 """
#                 Record every seg._ref_* that looks like a current:
#                   - startswith _ref_i
#                   - or contains 'ina', 'ik', 'il'
#                 """
#                 for attr in dir(seg):
#                     if not attr.startswith("_ref_"):
#                         continue
#                     name = attr[len("_ref_"):]
#                     lname = name.lower()
            
#                     is_currentish = (
#                         lname.startswith("i") or
#                         lname in ("ina", "ik", "il") or
#                         "ina" in lname or "ik" in lname or "il" in lname
#                     )
#                     if not is_currentish:
#                         continue
            
#                     ref = getattr(seg, attr)
#                     key = f"{tag}:{name}"
#                     _record_ref(diag_vecs, key, ref)


#             # Passive current
#             if include_pas_current:
#                 for var in pas_vars:
#                     if _has_ref(seg, var):
#                         _record_ref(diag["vecs"], f"{tag}:{var}", getattr(seg, f"_ref_{var}"))

#             # HH gating variables
#             if include_gating:
#                 for var in hh_gating:
#                     if _has_ref(seg, var):
#                         _record_ref(diag["vecs"], f"{tag}:{var}", getattr(seg, f"_ref_{var}"))


#     def list_seg_refs(seg, contains=None):
#         names = [a for a in dir(seg) if a.startswith("_ref_")]
#         if contains:
#             contains = contains.lower()
#             names = [n for n in names if contains in n.lower()]
#         return names
    
#     # Example: after the net is built
#     cell = net.ensure_cell(10000)
#     sec = cell.soma_sec
#     seg = sec(0.5)
    
#     print("REFS containing 'i':")
#     print(list_seg_refs(seg, contains="i")[:200])
    
#     print("\nREFS containing 'g':")
#     print(list_seg_refs(seg, contains="g")[:200])
    




    
#     # --------------------------------------------------------
#     # Synaptic currents: find syn objects on the Network
#     # This stack typically uses net.syns (list or dict) — handle both.
#     # --------------------------------------------------------
#     if include_syn_current:
#         syn_objs = []

#         for attr in ["syns", "_syns", "synapses", "_synapses", "syn_by_post"]:
#             if hasattr(net, attr):
#                 try:
#                     obj = getattr(net, attr)

#                     if isinstance(obj, (list, tuple)):
#                         syn_objs = list(obj)
#                         break

#                     if isinstance(obj, dict):
#                         tmp = []
#                         for v in obj.values():
#                             if isinstance(v, (list, tuple)):
#                                 tmp.extend(list(v))
#                             else:
#                                 tmp.append(v)
#                         syn_objs = tmp
#                         break
#                 except Exception:
#                     pass

#         if syn_objs:
#             rec = 0
#             for j, syn in enumerate(syn_objs):
#                 # Exp2Syn/ExpSyn expose i as RANGE variable, with _ref_i
#                 if hasattr(syn, "_ref_i"):
#                     v = h.Vector()
#                     v.record(syn._ref_i)
#                     diag["syn_vecs"][f"syn[{j}]:i_nA"] = v
#                     rec += 1
#             print(f"[diag] synapses found={len(syn_objs)}  recorded_i={rec}")
#         else:
#             print("[diag] no synapse container found on net (skipping syn.i)")

#     # --------------------------------------------------------
#     # IClamp currents: this stack typically uses net.iclamps
#     # --------------------------------------------------------
#     if include_iclamp_current:
#         clamp_objs = []

#         for attr in ["iclamps", "_iclamps", "iclamp_list", "_iclamp_list", "iclamp", "iclams", "_iclams"]:
#             if hasattr(net, attr):
#                 try:
#                     obj = getattr(net, attr)
#                     if isinstance(obj, (list, tuple)):
#                         clamp_objs = list(obj)
#                     else:
#                         clamp_objs = [obj]
#                     break
#                 except Exception:
#                     pass

#         if clamp_objs:
#             rec = 0
#             for j, stim in enumerate(clamp_objs):
#                 if hasattr(stim, "_ref_i"):
#                     v = h.Vector()
#                     v.record(stim._ref_i)
#                     diag["iclamp_vecs"][f"iclamp[{j}]:i_nA"] = v
#                     rec += 1
#             print(f"[diag] iclamps found={len(clamp_objs)}  recorded_i={rec}")
#         else:
#             print("[diag] no iclamp container found on net (skipping iclamp.i)")

#     # Metadata
#     if out_dir is not None:
#         diag["meta"]["out_dir"] = str(Path(out_dir).expanduser().resolve())

#     return diag


# def save_diagnostics_npz(diag, out_dir: str | Path, prefix="diag"):
#     out_dir = Path(out_dir).expanduser().resolve()
#     out_dir.mkdir(parents=True, exist_ok=True)
#     out_path = out_dir / f"{prefix}_channel_currents.npz"

#     data = {}

#     # core vectors
#     for k, v in diag["vecs"].items():
#         data[k] = np.asarray(v, dtype=float)

#     # syn vectors
#     for k, v in diag.get("syn_vecs", {}).items():
#         data["syn:" + k] = np.asarray(v, dtype=float)

#     # clamp vectors
#     for k, v in diag.get("iclamp_vecs", {}).items():
#         data["iclamp:" + k] = np.asarray(v, dtype=float)

#     # store metadata as an object array
#     data["__meta__"] = np.array([diag.get("meta", {})], dtype=object)

#     np.savez_compressed(out_path, **data)
#     print("[diag] saved:", out_path)
#     return out_path


# # ------------------ USAGE ------------------
# # IMPORTANT:
# #   By default, this block will NOT rerun the simulation.
# #   It will reuse diagnostics from the current run folder (out_dir).
# #   Set ENABLE_DIAG_RERUN = True only for an explicit diagnostic rerun.

# ENABLE_DIAG_RERUN = False

# DIAG_IDS = [10000, 10002, 10068, 11446, 11654, 10110]
# diag_holder = {}

# def _hook(net, cfg, out_dir):
#     diag_holder["diag"] = attach_channel_current_diagnostics(
#         net,
#         DIAG_IDS,
#         sites=("soma", "ais"),
#         out_dir=out_dir,
#         prefix="diag",
#         enable_imem=True,
#         include_gating=True,
#         include_hh_currents=True,
#         include_pas_current=True,
#         include_syn_current=True,
#         include_iclamp_current=True,
#     )

# base_out_dir = None
# if "out_dir" in globals() and out_dir is not None:
#     base_out_dir = Path(out_dir).expanduser().resolve()

# if ENABLE_DIAG_RERUN:
#     out_dir = run_walking_simulation(CONFIG, pre_run_hook=_hook)
#     npz_path = save_diagnostics_npz(diag_holder["diag"], out_dir, prefix="diag")
#     print("[diag] reran simulation. Diagnostics NPZ:", npz_path)
# else:
#     if base_out_dir is None:
#         npz_path = None
#         print("[diag] out_dir is not set. Run the main simulation cell first (Cell 33).")
#     else:
#         npz_path = base_out_dir / "diag_channel_currents.npz"
#         if npz_path.exists():
#             rec_path = base_out_dir / "records.csv"
#             if rec_path.exists() and npz_path.stat().st_mtime < rec_path.stat().st_mtime:
#                 print("[diag] existing diagnostics are older than records.csv:", npz_path)
#                 print("[diag] set ENABLE_DIAG_RERUN = True to refresh diagnostics for this run.")
#             else:
#                 print("[diag] using existing diagnostics:", npz_path)
#         else:
#             print("[diag] diagnostics file not found:", npz_path)
#             print("[diag] set ENABLE_DIAG_RERUN = True to generate it.")
# # =========================================================================================


In [ ]:
# import numpy as np
# from pathlib import Path

# npz_path = Path(npz_path)  # or Path(".../diag_channel_currents.npz")
# z = np.load(npz_path, allow_pickle=True)

# print("Keys:")
# for k in z.files[:50]:
#     print(" ", k)
# print("... total keys =", len(z.files))
# import matplotlib.pyplot as plt

# t = z["t_ms"]

# # pick one key, e.g. the first voltage trace key
# v_keys = [k for k in z.files if k.endswith(":v_mV")]
# print("Voltage traces:", v_keys[:10])

# k = v_keys[0]
# v = z[k]

# plt.figure()
# plt.plot(t, v)
# plt.title(k)
# plt.xlabel("Time (ms)")
# plt.ylabel("V (mV)")
# plt.tight_layout()
# plt.show()
# # Clamp currents
# iclamp_keys = [k for k in z.files if k.startswith("iclamp:")]
# print("IClamp keys:", iclamp_keys)

# # Synaptic currents
# syn_keys = [k for k in z.files if k.startswith("syn:")]
# print("Syn keys:", syn_keys[:10], " ... total", len(syn_keys))

# # Example: plot first iclamp current
# if iclamp_keys:
#     i = z[iclamp_keys[0]]
#     plt.figure()
#     plt.plot(t, i)
#     plt.title(iclamp_keys[0])
#     plt.xlabel("Time (ms)")
#     plt.ylabel("IClamp i (nA)")
#     plt.tight_layout()
#     plt.show()

# # Example: plot one syn current
# if syn_keys:
#     i = z[syn_keys[0]]
#     plt.figure()
#     plt.plot(t, i)
#     plt.title(syn_keys[0])
#     plt.xlabel("Time (ms)")
#     plt.ylabel("Syn i (nA)")
#     plt.tight_layout()
#     plt.show()


In [ ]:
# iclamp_keys = [k for k in z.files if "iclamp" in k.lower() or k.lower().startswith("iclamp:") or "clamp" in k.lower()]
# print("Found clamp-like keys:", len(iclamp_keys))
# print(iclamp_keys[:20])


In [ ]:
# # -------- set these to the actual stimulation values ----------
# STIM_DELAY_MS = 10.0
# STIM_DUR_MS   = 0.3     # use the actual duration (ms)

# t = z["t_ms"]
# t0 = STIM_DELAY_MS
# t1 = STIM_DELAY_MS + STIM_DUR_MS
# post_start = t1 + 0.5   # start checking 0.5 ms after pulse ends
# post_end   = t[-1]      # end of sim

# mask_post = (t >= post_start) & (t <= post_end)

# def _get(key):
#     return z[key].astype(float)

# def _find_key_contains(sub):
#     sub = sub.lower()
#     return [k for k in z.files if sub in k.lower()]

# # --- 2A) Total synaptic current across ALL synapses ---
# syn_keys = [k for k in z.files if k.startswith("syn:")]
# print("syn currents:", len(syn_keys))

# if syn_keys:
#     I_syn_total = np.zeros_like(t, dtype=float)
#     for k in syn_keys:
#         I_syn_total += _get(k)

#     print("[POST] mean|I_syn_total| (nA) =", float(np.mean(np.abs(I_syn_total[mask_post]))))
# else:
#     I_syn_total = None
#     print("No syn currents found.")

# # --- 2B) For each neuron, compare mean absolute ionic currents after stim ---
# def summarize_cell(nid, site="soma"):
#     # pick the first matching keys for that nid+site
#     prefix = f"{nid}:{site}:"
#     v_keys  = [k for k in z.files if k.startswith(prefix) and k.endswith(":v_mV")]
#     ina_keys= [k for k in z.files if k.startswith(prefix) and k.endswith(":ina")]
#     ik_keys = [k for k in z.files if k.startswith(prefix) and k.endswith(":ik")]
#     im_keys = [k for k in z.files if k.startswith(prefix) and (k.endswith(":i_membrane_") or k.endswith(":i_membrane"))]

#     if not v_keys:
#         return None

#     v   = _get(v_keys[0])
#     ina = _get(ina_keys[0]) if ina_keys else None
#     ik  = _get(ik_keys[0])  if ik_keys  else None
#     im  = _get(im_keys[0])  if im_keys  else None

#     out = {
#         "nid": nid,
#         "site": site,
#         "V_post_mean": float(np.mean(v[mask_post])),
#         "V_post_max":  float(np.max(v[mask_post])),
#     }
#     if ina is not None: out["mean|ina|_post"] = float(np.mean(np.abs(ina[mask_post])))
#     if ik  is not None: out["mean|ik|_post"]  = float(np.mean(np.abs(ik[mask_post])))
#     if im  is not None: out["mean|i_mem|_post"]= float(np.mean(np.abs(im[mask_post])))
#     return out

# for nid in DIAG_IDS:
#     s = summarize_cell(nid, "soma")
#     a = summarize_cell(nid, "ais")
#     print(s)
#     print(a)
#     print("-"*60)


In [ ]:
# import matplotlib.pyplot as plt

# nid = 10000
# site = "soma"

# # pick keys
# prefix = f"{nid}:{site}:"
# v_key   = [k for k in z.files if k.startswith(prefix) and k.endswith(":v_mV")][0]
# ina_key = [k for k in z.files if k.startswith(prefix) and k.endswith(":ina")][0]
# ik_key  = [k for k in z.files if k.startswith(prefix) and k.endswith(":ik")][0]
# im_key  = [k for k in z.files if k.startswith(prefix) and k.endswith(":i_membrane_")][0]

# v   = z[v_key].astype(float)
# ina = z[ina_key].astype(float)
# ik  = z[ik_key].astype(float)
# im  = z[im_key].astype(float)

# plt.figure()
# plt.plot(t, v)
# plt.axvline(t0, linestyle="--")
# plt.axvline(t1, linestyle="--")
# plt.title(v_key)
# plt.xlabel("ms"); plt.ylabel("mV")
# plt.tight_layout()
# plt.show()

# plt.figure()
# plt.plot(t, ina, label="ina")
# plt.plot(t, ik,  label="ik")
# plt.plot(t, im,  label="i_membrane_")
# if I_syn_total is not None:
#     plt.plot(t, I_syn_total, label="I_syn_total")
# plt.axvline(t0, linestyle="--")
# plt.axvline(t1, linestyle="--")
# plt.legend()
# plt.xlabel("ms"); plt.ylabel("nA (or NEURON units)")
# plt.tight_layout()
# plt.show()


In [ ]:
# import matplotlib.pyplot as plt
# t = z["t_ms"]
# plt.figure()
# plt.plot(t, I_syn_total)
# plt.axvline(10.0, linestyle="--")
# plt.axvline(10.0 + 0.3, linestyle="--")
# plt.xlabel("Time (ms)")
# plt.ylabel("Total syn current (nA)")
# plt.title("Total synaptic current across all synapses")
# plt.tight_layout()
# plt.show()


In [ ]:
# # Find the top syn currents by post-window mean abs current
# vals = []
# for k in syn_keys:
#     i = z[k].astype(float)
#     vals.append((float(np.mean(np.abs(i[mask_post]))), k))

# vals.sort(reverse=True)
# print("Top 20 synapses by mean|i| in post window:")
# for m, k in vals[:20]:
#     print(f"{m:.6f}  {k}")


In [ ]:
# import numpy as np

# t = z["t_ms"].astype(float)

# # pick a driven neuron + site of interest
# v_key = "10000:soma:cell10000_n149@0.500:v_mV"
# v = z[v_key].astype(float)

# # spike time = time of max V
# t_peak = float(t[np.argmax(v)])
# print("t_peak(ms) =", t_peak)

# # define post window as 2 ms after the peak to end
# post_start = t_peak + 2.0
# post_end = float(t[-1])
# mask_post = (t >= post_start) & (t <= post_end)

# # recompute syn stats in the correct window
# syn_keys = [k for k in z.files if k.startswith("syn:")]
# I_syn_total = np.zeros_like(t)
# for k in syn_keys:
#     I_syn_total += z[k].astype(float)

# print("mean|I_syn_total| post (nA) =", float(np.mean(np.abs(I_syn_total[mask_post]))))
# print("max|I_syn_total| post (nA)  =", float(np.max(np.abs(I_syn_total[mask_post]))))
